# SLM-RAG Step 3: why does the small model fail to read evidence it has?

Kaggle: **GPU T4**, **Internet On**. Nothing to upload. Safe to re-run: finished runs are skipped.

Step 2 showed two gaps: finding the right passages, and reading them. Even with the gold passages the 1.5B model gets only about 0.59.
This notebook tests what causes the reading gap. Each test changes ONE thing:

| Test | Question |
|---|---|
| A. Number of passages (k = 1, 3, 5, 10, 20) | Do more passages help the answer or just add noise? |
| B. Passage order and position | Does it matter if the gold passage is first, last, or reversed? |
| C. Prompt | Does "use your own knowledge if the context does not help" reduce context capture? Does short step-by-step reasoning raise the 0.59? |
| D. Error types | When it is wrong, did it copy the wrong span, give up, or make something up? |
| E. Other model families | Is the pattern only a Qwen thing? |

Sections: 0-2 setup, 3 smoke test, 4 tests A-C on Qwen, 5 other families, 6 tables, 7 zip.

## 0. Environment

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.chdir("/kaggle/working")
import torch
assert torch.cuda.is_available(), "Turn on a GPU: Settings > Accelerator"
print(torch.__version__, torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

2.11.0+cu128 Tesla T4 15.6 GB


In [2]:
!pip install -q bitsandbytes sentence-transformers datasets accelerate scipy pandas pytest

## 1. Write the package to disk

In [3]:
%%writefile slmrag_core.py
"""slmrag_core: measurement-fixed evaluation core for the SLM-RAG study (Steps 1-3).

Dependency-light on purpose (numpy + scipy only) so every function here is
unit-tested without a GPU. Model code lives in slmrag_hf.py, dataset code in
slmrag_data.py, and both plug into `evaluate()` below through plain callables.

Sections
  1. Answer normalisation and metrics (EM, F1, containment accuracy, PubMedQA labels)
  2. Generation post-processing (stop handling, answer extraction)
  3. Prompts and context budgeting
  4. Retrieval (BM25, dense, hybrid RRF) and retrieval metrics
  5. Statistics (bootstrap CI, paired tests, Holm, McNemar, power, Kendall tau)
  6. evaluate(): one setting, one model, per-example records
"""
from __future__ import annotations

import collections
import math
import re
import string
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
from scipy import sparse, stats as sps

# --------------------------------------------------------------------------
# 1. Normalisation and metrics
# --------------------------------------------------------------------------
_PUNCT = set(string.punctuation)
_ARTICLES = re.compile(r"\b(a|an|the)\b", re.I)


def normalize_answer(s: str) -> str:
    """SQuAD-style normalisation: lowercase, strip punctuation and articles."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in _PUNCT)
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


def _as_list(golds) -> List[str]:
    return [golds] if isinstance(golds, str) else list(golds)


def exact_match(pred: str, golds) -> float:
    p = normalize_answer(pred)
    return float(any(p == normalize_answer(g) for g in _as_list(golds)))


def _f1_single(pred: str, gold: str) -> float:
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not pt or not gt:
        return float(pt == gt)
    common = collections.Counter(pt) & collections.Counter(gt)
    n = sum(common.values())
    if n == 0:
        return 0.0
    prec, rec = n / len(pt), n / len(gt)
    return 2 * prec * rec / (prec + rec)


def f1_score(pred: str, golds) -> float:
    """Token F1, maximum over gold aliases."""
    return max(_f1_single(pred, g) for g in _as_list(golds))


def contains_answer(text: str, golds) -> bool:
    """True if any normalised gold answer occurs in `text` on token boundaries."""
    t = f" {normalize_answer(text)} "
    for g in _as_list(golds):
        ng = normalize_answer(g)
        if ng and f" {ng} " in t:
            return True
    return False


def acc_contain(pred: str, golds) -> float:
    """Containment accuracy (standard for NQ-Open / PopQA style evaluation)."""
    return float(contains_answer(pred, golds))


_LABEL_RE = re.compile(r"\b(yes|no|maybe)\b", re.I)
LABELS = ("yes", "no", "maybe")


def extract_label(pred: str) -> Optional[str]:
    """First yes/no/maybe word in the (cleaned) prediction, else None."""
    m = _LABEL_RE.search(pred or "")
    return m.group(1).lower() if m else None


def label_metrics(preds: Sequence[Optional[str]], golds: Sequence[str]) -> Dict[str, float]:
    """Accuracy and macro-F1 for PubMedQA decisions. Unparsable predictions count as wrong."""
    assert len(preds) == len(golds)
    acc = float(np.mean([p == g for p, g in zip(preds, golds)])) if golds else 0.0
    f1s = []
    for lab in LABELS:
        tp = sum(p == lab and g == lab for p, g in zip(preds, golds))
        fp = sum(p == lab and g != lab for p, g in zip(preds, golds))
        fn = sum(p != lab and g == lab for p, g in zip(preds, golds))
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return {"accuracy": acc, "macro_f1": float(np.mean(f1s))}


def is_yesno_gold(answers) -> bool:
    """True for HotpotQA-style comparison questions whose gold answer is just yes/no.
    Containment accuracy is inflated on these (a 'yes' or 'no' appears inside many outputs)."""
    return any(normalize_answer(a) in ("yes", "no") for a in _as_list(answers))


def calibrate_labels(probs: Sequence[Dict[str, float]]) -> List[str]:
    """Label-free prior matching for PubMedQA: divide each class probability by that class's
    mean probability over the evaluated set, then take the argmax. Uses NO gold labels, so it
    removes a constant 'always say yes' bias without touching the test labels."""
    prior = {l: max(float(np.mean([p[l] for p in probs])), 1e-12) for l in LABELS}
    return [max(LABELS, key=lambda l: p[l] / prior[l]) for p in probs]


# --------------------------------------------------------------------------
# 2. Generation post-processing
# --------------------------------------------------------------------------
_STOP = re.compile(
    r"(\n|\bQuestion\s*:|\bQ\s*:|\bContext\s*:|\bPassages?\b\s*[\[:(]|\bExplanation\s*:|"
    r"<\|im_end\|>|<\|endoftext\|>|<\|eot_id\|>|###)",
    re.I,
)
_PREFIX = re.compile(r"^\s*(final\s+answer|answer|a)\s*[:\-]\s*", re.I)


def clean_generation(raw: str, max_words: Optional[int] = None) -> str:
    """Reduce a raw generation to the answer span.

    Cuts at the first newline or at a hallucinated follow-up ("Question:", "Context:" ...),
    removes "Answer:" prefixes, surrounding quotes and a trailing full stop.
    """
    s = (raw or "").strip()
    m = _STOP.search(s)
    if m:
        s = s[: m.start()]
    prev = None
    while prev != s:  # strip repeated prefixes
        prev, s = s, _PREFIX.sub("", s).strip()
    s = s.strip().strip("\"'`*").strip()
    if s.endswith(".") and not re.search(r"\b[A-Z]\.$", s):
        s = s[:-1].strip()
    if max_words is not None:
        s = " ".join(s.split()[:max_words])
    return s


# --------------------------------------------------------------------------
# 3. Prompts and context budgeting
# --------------------------------------------------------------------------
SYSTEM_SHORT = (
    "You are a precise question-answering assistant. Reply with only the answer, "
    "as a short phrase of at most five words. Do not explain. Do not repeat the question."
)
SYSTEM_YESNO = (
    "You are a precise biomedical question-answering assistant. Reply with exactly one word: "
    "yes, no, or maybe. Do not explain."
)


SYSTEM_OWN = (
    "You are a precise question-answering assistant. Use the context if it helps. If the context does not "
    "contain the answer, answer from your own knowledge instead. Reply with only the answer, as a short "
    "phrase of at most five words. Do not explain."
)
SYSTEM_COT = (
    "You are a careful question-answering assistant. Think step by step in at most three short sentences, "
    "using the context if one is given and your own knowledge if the context is missing or unhelpful. "
    "Then write a last line exactly in the form 'Final answer: <short phrase of at most five words>'."
)
VARIANTS = {"base": SYSTEM_SHORT, "own": SYSTEM_OWN, "cot": SYSTEM_COT}
_FINAL = re.compile(r"final\s+answer\s*[:\-]", re.I)


def extract_final(raw: str) -> Tuple[str, bool]:
    """Answer for the 'cot' prompt: text after the LAST 'Final answer:'; if the model never wrote it,
    fall back to the last non-empty line. Returns (answer, found_marker)."""
    s = (raw or "").strip()
    ms = list(_FINAL.finditer(s))
    if ms:
        return clean_generation(s[ms[-1].end():]), True
    lines = [l for l in s.splitlines() if l.strip()]
    return clean_generation(lines[-1] if lines else ""), False


def format_passages(passages: Sequence[str]) -> str:
    return "\n".join(f"[{i + 1}] {p.strip()}" for i, p in enumerate(passages))


def truncate_passages(
    passages: Sequence[str],
    budget: int,
    len_fn: Callable[[str], int] = lambda s: len(s.split()),
) -> List[str]:
    """Keep whole passages in rank order until `budget` is spent.

    If even the first passage exceeds the budget it is cut by words. The same
    budget must be used for every retriever so context length is not a confound.
    """
    out, used = [], 0
    for p in passages:
        n = len_fn(p)
        if used + n <= budget:
            out.append(p)
            used += n
        elif not out:
            out.append(" ".join(p.split()[:budget]))
            break
        else:
            break
    return out


def build_messages(question: str, passages: Optional[Sequence[str]], task: str = "qa",
                   variant: str = "base") -> List[dict]:
    """Chat messages. passages=None means closed-book. `variant` picks the system prompt for QA
    (base | own = 'use your own knowledge if the context does not help' | cot = brief reasoning)."""
    system = SYSTEM_YESNO if task == "yesno" else VARIANTS[variant]
    if passages:
        user = f"Context:\n{format_passages(passages)}\n\nQuestion: {question.strip()}\nAnswer:"
    else:
        user = f"Question: {question.strip()}\nAnswer:"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


# --------------------------------------------------------------------------
# 4. Retrieval
# --------------------------------------------------------------------------
_STOPWORDS = frozenset(
    "a an the of in on at to for from by with and or is are was were be been it its this that "
    "as which who whom what when where how do does did".split()
)
_TOK = re.compile(r"\w+")


def tokenize(text: str) -> List[str]:
    return [t for t in _TOK.findall(text.lower()) if t not in _STOPWORDS]


def chunk_words(text: str, size: int = 128, overlap: int = 0) -> List[str]:
    w = text.split()
    if not w:
        return []
    step = max(1, size - overlap)
    return [" ".join(w[i : i + size]) for i in range(0, max(1, len(w) - overlap), step)]


class BM25:
    """Okapi BM25 (Anserini defaults k1=0.9, b=0.4) on a precomputed sparse weight matrix."""

    def __init__(self, docs: Sequence[str], k1: float = 0.9, b: float = 0.4):
        self.n = len(docs)
        vocab: Dict[str, int] = {}
        rows, cols, tfs, dl = [], [], [], np.zeros(self.n)
        for i, d in enumerate(docs):
            toks = tokenize(d)
            dl[i] = len(toks)
            for t, c in collections.Counter(toks).items():
                rows.append(i)
                cols.append(vocab.setdefault(t, len(vocab)))
                tfs.append(c)
        self.vocab = vocab
        rows, cols, tfs = np.array(rows), np.array(cols), np.array(tfs, dtype=float)
        df = np.bincount(cols, minlength=len(vocab)).astype(float)
        idf = np.log(1.0 + (self.n - df + 0.5) / (df + 0.5))
        avgdl = max(dl.mean(), 1e-9)
        w = idf[cols] * tfs * (k1 + 1) / (tfs + k1 * (1 - b + b * dl[rows] / avgdl))
        self.W = sparse.csc_matrix((w, (rows, cols)), shape=(self.n, len(vocab)))

    def scores(self, query: str) -> np.ndarray:
        idx = sorted({self.vocab[t] for t in tokenize(query) if t in self.vocab})
        if not idx:
            return np.zeros(self.n)
        return np.asarray(self.W[:, idx].sum(axis=1)).ravel()

    def search(self, query: str, k: int = 10) -> List[Tuple[int, float]]:
        s = self.scores(query)
        k = min(k, self.n)
        top = np.argpartition(-s, k - 1)[:k]
        top = top[np.argsort(-s[top], kind="stable")]
        return [(int(i), float(s[i])) for i in top]


class DenseRetriever:
    """Exact inner-product search over L2-normalised embeddings (numpy, no FAISS needed).

    encode_fn(list[str]) -> float array [n, d]. Use make_st_encoder() for sentence-transformers.
    Pass `embeddings=` to reuse a cached matrix (encoding a large pool is the slow part).
    """

    def __init__(self, docs: Sequence[str], encode_fn: Callable[[List[str]], np.ndarray],
                 query_prefix: str = "", doc_prefix: str = "", batch: int = 256,
                 embeddings: Optional[np.ndarray] = None):
        self.encode, self.qp = encode_fn, query_prefix
        if embeddings is not None:
            assert len(embeddings) == len(docs), "cached embeddings do not match the pool"
            self.E = np.asarray(embeddings, dtype=np.float32)
            return
        embs = [self._norm(encode_fn([doc_prefix + d for d in docs[i : i + batch]]))
                for i in range(0, len(docs), batch)]
        self.E = np.vstack(embs).astype(np.float32)

    @staticmethod
    def _norm(x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=np.float32)
        return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-9)

    def search(self, query: str, k: int = 10) -> List[Tuple[int, float]]:
        return self.search_batch([query], k)[0]

    def search_batch(self, queries: Sequence[str], k: int = 10, block: int = 256) -> List[List[Tuple[int, float]]]:
        out: List[List[Tuple[int, float]]] = []
        k = min(k, len(self.E))
        for i in range(0, len(queries), block):
            q = self._norm(self.encode([self.qp + x for x in queries[i : i + block]]))
            S = q @ self.E.T
            top = np.argpartition(-S, k - 1, axis=1)[:, :k]
            for r in range(len(top)):
                t = top[r][np.argsort(-S[r, top[r]], kind="stable")]
                out.append([(int(j), float(S[r, j])) for j in t])
        return out


def make_st_encoder(model_name: str = "intfloat/e5-base-v2", device: Optional[str] = None, batch_size: int = 128):
    """sentence-transformers encoder (fp16 on GPU, 256-token cap). E5 needs prefixes:
    pass query_prefix='query: ', doc_prefix='passage: '."""
    from sentence_transformers import SentenceTransformer  # lazy: heavy import

    m = SentenceTransformer(model_name, device=device)
    m.max_seq_length = min(getattr(m, "max_seq_length", 512) or 512, 256)
    if str(m.device).startswith("cuda"):
        m.half()
    return lambda texts: m.encode(texts, batch_size=batch_size, convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False)


def rrf(rank_lists: Sequence[Sequence[int]], k: int = 60, top: int = 10) -> List[Tuple[int, float]]:
    """Reciprocal rank fusion over several ranked lists of doc ids."""
    score: Dict[int, float] = collections.defaultdict(float)
    for lst in rank_lists:
        for r, d in enumerate(lst):
            score[d] += 1.0 / (k + r + 1)
    return sorted(score.items(), key=lambda kv: (-kv[1], kv[0]))[:top]


class HybridRRF:
    def __init__(self, a, b, depth: int = 100, k: int = 60):
        self.a, self.b, self.depth, self.k = a, b, depth, k

    def search(self, query: str, k: int = 10) -> List[Tuple[int, float]]:
        la = [d for d, _ in self.a.search(query, self.depth)]
        lb = [d for d, _ in self.b.search(query, self.depth)]
        return rrf([la, lb], k=self.k, top=k)


def retrieval_metrics(retrieved_texts: Sequence[Sequence[str]], golds: Sequence,
                      ks: Sequence[int] = (1, 5, 10)) -> Dict[str, float]:
    """Answer-containment retrieval quality: hit@k and MRR@max(k). Queries with no answer string are skipped."""
    hits = {k: [] for k in ks}
    rr = []
    kmax = max(ks)
    for texts, g in zip(retrieved_texts, golds):
        flags = [contains_answer(t, g) for t in texts[:kmax]]
        for k in ks:
            hits[k].append(any(flags[:k]))
        rr.append(next((1.0 / (i + 1) for i, f in enumerate(flags) if f), 0.0))
    out = {f"hit@{k}": float(np.mean(v)) if v else 0.0 for k, v in hits.items()}
    out[f"mrr@{kmax}"] = float(np.mean(rr)) if rr else 0.0
    return out


def id_recall_at_k(retrieved_ids: Sequence[Sequence], gold_ids: Sequence[Sequence], k: int) -> float:
    """Mean fraction of gold documents found in the top-k (queries without gold ids are skipped)."""
    vals = [len(set(r[:k]) & set(g)) / len(set(g)) for r, g in zip(retrieved_ids, gold_ids) if g]
    return float(np.mean(vals)) if vals else float("nan")


def all_gold_at_k(retrieved_ids: Sequence[Sequence], gold_ids: Sequence[Sequence], k: int) -> float:
    """Fraction of queries whose gold documents are ALL in the top-k (the multi-hop 'complete evidence' rate)."""
    vals = [set(g) <= set(r[:k]) for r, g in zip(retrieved_ids, gold_ids) if g]
    return float(np.mean(vals)) if vals else float("nan")


# --------------------------------------------------------------------------
# 5. Statistics
# --------------------------------------------------------------------------
def bootstrap_ci(x, B: int = 10000, alpha: float = 0.05, seed: int = 0) -> Tuple[float, float, float]:
    """(mean, lo, hi) percentile bootstrap, chunked to bound memory."""
    x = np.asarray(x, dtype=float)
    n, rng = len(x), np.random.default_rng(seed)
    means = np.empty(B)
    step = max(1, int(2e7 // max(n, 1)))
    for s in range(0, B, step):
        e = min(B, s + step)
        means[s:e] = x[rng.integers(0, n, (e - s, n))].mean(axis=1)
    lo, hi = np.quantile(means, [alpha / 2, 1 - alpha / 2])
    return float(x.mean()), float(lo), float(hi)


def paired_compare(a, b, binary: bool = False, B: int = 10000, seed: int = 0) -> Dict[str, float]:
    """Paired comparison of per-example scores a vs b (same examples, same order).

    Returns mean difference (a-b), bootstrap CI, paired t, Wilcoxon, Cohen's dz and,
    for 0/1 scores, the exact McNemar p-value (the correct test for accuracy).
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    assert a.shape == b.shape, "paired arrays must align"
    d = a - b
    mean, lo, hi = bootstrap_ci(d, B=B, seed=seed)
    sd = d.std(ddof=1) if len(d) > 1 else 0.0
    out = {"n": len(d), "mean_diff": mean, "ci_lo": lo, "ci_hi": hi,
           "dz": float(mean / sd) if sd > 0 else 0.0, "sd_diff": float(sd)}
    out["p_t"] = float(sps.ttest_rel(a, b).pvalue) if sd > 0 else 1.0
    nz = d[d != 0]
    out["p_wilcoxon"] = float(sps.wilcoxon(nz).pvalue) if len(nz) > 0 else 1.0
    if binary:
        n01, n10 = int(((a == 0) & (b == 1)).sum()), int(((a == 1) & (b == 0)).sum())
        out["p_mcnemar"] = float(sps.binomtest(n10, n10 + n01, 0.5).pvalue) if n10 + n01 > 0 else 1.0
        out["discordant"] = n10 + n01
    return out


def holm(pvals: Sequence[float]) -> List[float]:
    """Holm-Bonferroni adjusted p-values (monotone, capped at 1)."""
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj, running = np.empty_like(p), 0.0
    m = len(p)
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj.tolist()


def mde_paired(n: int, sd_diff: float, alpha: float = 0.05, power: float = 0.8) -> float:
    """Minimum detectable mean difference for a paired test (normal approximation)."""
    z = sps.norm.ppf(1 - alpha / 2) + sps.norm.ppf(power)
    return float(z * sd_diff / math.sqrt(n))


def n_required(delta: float, sd_diff: float, alpha: float = 0.05, power: float = 0.8) -> int:
    z = sps.norm.ppf(1 - alpha / 2) + sps.norm.ppf(power)
    return int(math.ceil((z * sd_diff / delta) ** 2))


def kendall_tau(x: Sequence[float], y: Sequence[float]) -> Tuple[float, float]:
    r = sps.kendalltau(x, y)
    return float(r.statistic), float(r.pvalue)


# --------------------------------------------------------------------------
# 6. evaluate(): one setting, one model
# --------------------------------------------------------------------------
def evaluate(
    records: Sequence[dict],
    generate_fn: Callable[[List[List[dict]]], List[Tuple[str, bool]]],
    retrieve_fn: Optional[Callable[[dict], List[str]]] = None,
    context_budget: int = 600,
    len_fn: Callable[[str], int] = lambda s: len(s.split()),
    max_words: Optional[int] = None,
    label_fn: Optional[Callable[[List[List[dict]]], List[Dict[str, float]]]] = None,
    variant: str = "base",
) -> Tuple[List[dict], Dict[str, float]]:
    """Run generation for every record and score it.

    label_fn (optional): messages -> {label: probability} for yes/no/maybe records. When given,
    PubMedQA-style records are scored by their next-token probabilities instead of free text,
    which removes parsing noise and exposes the model's yes-bias (see calibrate_labels).

    records: dicts with id, question, answers (list[str]), task ('qa'|'yesno'), label (yesno only)
    generate_fn: list of chat-message lists -> list of (raw_text, hit_max_new_tokens)
    retrieve_fn: record -> ranked passage strings (None = closed-book)
    Returns (per-example rows, summary). Rows keep the RAW output so any post-processing
    can be audited later.
    """
    prompts, ctxs = [], []
    for r in records:
        ps = retrieve_fn(r) if retrieve_fn else None
        ps = truncate_passages(ps, context_budget, len_fn) if ps else None
        ctxs.append(ps)
        prompts.append(build_messages(r["question"], ps, r.get("task", "qa"), variant))
    use_probs = label_fn is not None and all(r.get("task", "qa") == "yesno" for r in records)
    if use_probs:
        probs = label_fn(prompts)
        outs = [(max(LABELS, key=lambda l: p[l]), False) for p in probs]
        cal = calibrate_labels(probs)
    else:
        outs = generate_fn(prompts)
        probs, cal = [None] * len(records), [None] * len(records)
    assert len(outs) == len(records)

    rows = []
    for r, ps, (raw, hit_max), pr, cl in zip(records, ctxs, outs, probs, cal):
        if variant == "cot" and r.get("task", "qa") == "qa":
            pred, found = extract_final(raw)
        else:
            pred, found = clean_generation(raw, max_words=max_words), True
        row = {"id": r["id"], "task": r.get("task", "qa"), "raw": raw, "pred": pred,
               "hit_max": bool(hit_max), "n_ctx": len(ps) if ps else 0, "final_found": found}
        if r.get("task", "qa") == "yesno":
            lab = extract_label(pred)
            row.update(label=r["label"], pred_label=lab, acc=float(lab == r["label"]))
            if pr is not None:
                row.update(probs=pr, pred_cal=cl, acc_cal=float(cl == r["label"]))
        else:
            row.update(answers=r["answers"], em=exact_match(pred, r["answers"]),
                       f1=f1_score(pred, r["answers"]), acc=acc_contain(pred, r["answers"]),
                       raw_f1=f1_score(raw, r["answers"]), raw_acc=acc_contain(raw, r["answers"]),
                       ctx_has_answer=bool(ps) and any(contains_answer(p, r["answers"]) for p in ps),
                       yn_q=is_yesno_gold(r["answers"]),
                       pred_in_ctx=bool(ps) and bool(pred) and any(contains_answer(p, [pred]) for p in ps),
                       qtype=r.get("qtype"), level=r.get("level"))
        rows.append(row)
    return rows, summarize(rows)


def summarize(rows: Sequence[dict]) -> Dict[str, float]:
    s: Dict[str, float] = {"n": len(rows)}
    qa = [r for r in rows if r["task"] == "qa"]
    yn = [r for r in rows if r["task"] == "yesno"]
    if qa:
        for k in ("em", "f1", "acc"):
            m, lo, hi = bootstrap_ci([r[k] for r in qa], B=2000)
            s.update({k: m, f"{k}_lo": lo, f"{k}_hi": hi})
        sub = [r for r in qa if not r.get("yn_q")]
        s["n_nonyn"] = len(sub)
        if sub and len(sub) < len(qa):
            for k in ("em", "f1", "acc"):
                s[f"{k}_nonyn"] = float(np.mean([r[k] for r in sub]))
        s["raw_f1"] = float(np.mean([r["raw_f1"] for r in qa]))
        s["mean_pred_words"] = float(np.mean([len(r["pred"].split()) for r in qa]))
        s["mean_raw_words"] = float(np.mean([len(r["raw"].split()) for r in qa]))
    if yn:
        s.update({f"yn_{k}": v for k, v in label_metrics(
            [r["pred_label"] for r in yn], [r["label"] for r in yn]).items()})
        s["yn_unparsed"] = float(np.mean([r["pred_label"] is None for r in yn]))
        gold = [r["label"] for r in yn]
        s["yn_majority_acc"] = max(gold.count(l) for l in LABELS) / len(gold)
        for l in LABELS:
            s[f"yn_pred_{l}"] = float(np.mean([r["pred_label"] == l for r in yn]))
        if all("pred_cal" in r for r in yn):
            s.update({f"yn_cal_{k}": v for k, v in label_metrics([r["pred_cal"] for r in yn], gold).items()})
    s["final_found_rate"] = float(np.mean([r.get("final_found", True) for r in rows])) if rows else 1.0
    s["hit_max_rate"] = float(np.mean([r["hit_max"] for r in rows])) if rows else 0.0
    s["empty_rate"] = float(np.mean([not r["pred"] for r in rows])) if rows else 0.0
    return s


def sanity_warnings(summaries: Dict[str, Dict[str, float]]) -> List[str]:
    """Objective red flags across settings (keys are setting names such as 'closed', 'oracle')."""
    w = []
    for name, s in summaries.items():
        if s.get("hit_max_rate", 0) > 0.02:
            w.append(f"[{name}] {s['hit_max_rate']:.1%} of generations hit max_new_tokens: stop handling is not working.")
        if s.get("empty_rate", 0) > 0.02:
            w.append(f"[{name}] {s['empty_rate']:.1%} empty predictions after cleaning.")
        if s.get("mean_pred_words", 0) > 8:
            w.append(f"[{name}] mean cleaned answer is {s['mean_pred_words']:.1f} words: prompt is not producing short answers.")
        if s.get("raw_f1") is not None and "f1" in s and s["f1"] > 0 and s["f1"] / max(s["raw_f1"], 1e-9) > 3:
            w.append(f"[{name}] cleaned F1 is >3x raw F1: results depend heavily on post-processing, report both.")
        if s.get("yn_unparsed", 0) > 0.05:
            w.append(f"[{name}] {s['yn_unparsed']:.1%} of PubMedQA outputs contain no yes/no/maybe.")
    if "closed" in summaries and "oracle" in summaries:
        c, o = summaries["closed"], summaries["oracle"]
        key = "acc" if "acc" in c else "yn_accuracy"
        if key in c and key in o and o[key] <= c[key]:
            w.append("oracle context does not beat closed-book: the generation or prompt path is still broken.")
    return w

Overwriting slmrag_core.py


In [4]:
%%writefile slmrag_data.py
"""Dataset loaders for Steps 1-2. Pure transform functions are unit-tested; the HF download
calls are not (no network in the dev sandbox), so run `python slmrag_data.py --check` once on
your machine: it prints one record per dataset and fails loudly if a schema differs.

Record format (all datasets)
  id, question, task ('qa'|'yesno'), answers (list[str]; qa), label (yesno),
  gold_passages (list[str]), gold_ids (list[str])
"""
from __future__ import annotations

import json
import random
import sys
from typing import Dict, List, Optional, Tuple


# ---- pure transforms -------------------------------------------------------
def nq_record(i: int, row: dict) -> dict:
    q = row["question"].strip()
    if not q.endswith("?"):  # nq_open questions are lower-case fragments without a question mark
        q += "?"
    return {"id": f"nq-{i}", "question": q, "task": "qa", "answers": list(row["answer"]),
            "gold_passages": [], "gold_ids": []}


def hotpot_paragraphs(row: dict) -> List[Tuple[str, str]]:
    ctx = row["context"]
    return [(t, f"{t}: {' '.join(s).strip()}") for t, s in zip(ctx["title"], ctx["sentences"])]


def hotpot_record(row: dict) -> dict:
    paras = dict(hotpot_paragraphs(row))
    gold_titles = list(dict.fromkeys(row["supporting_facts"]["title"]))  # unique, ordered
    return {"id": f"hp-{row['id']}", "question": row["question"].strip(), "task": "qa",
            "answers": [row["answer"]], "gold_passages": [paras[t] for t in gold_titles if t in paras],
            "gold_ids": [t for t in gold_titles if t in paras],
            "qtype": row.get("type"), "level": row.get("level")}


def pubmed_record(row: dict) -> dict:
    ctx = " ".join(row["context"]["contexts"]).strip()
    return {"id": f"pm-{row['pubid']}", "question": row["question"].strip(), "task": "yesno",
            "label": row["final_decision"].strip().lower(), "answers": [],
            "gold_passages": [ctx], "gold_ids": [str(row["pubid"])]}


# ---- loaders ---------------------------------------------------------------
def _load(candidates, **kw):
    from datasets import load_dataset  # lazy
    last = None
    for c in candidates:
        try:
            return load_dataset(*c, **kw)
        except Exception as e:  # dataset renamed / script-based loader removed, try next id
            last = e
    raise RuntimeError(f"could not load any of {candidates}: {last}")


def _sample(idx: List[int], n: Optional[int], seed: int) -> List[int]:
    idx = list(idx)
    random.Random(seed).shuffle(idx)
    return idx[:n] if n else idx


def load_nq(n: int, seed: int = 42, split: str = "validation") -> Tuple[List[dict], Optional[List[dict]]]:
    ds = _load([("google-research-datasets/nq_open",), ("nq_open",)], split=split)
    return [nq_record(i, ds[i]) for i in _sample(range(len(ds)), n, seed)], None  # no pool: needs a Wikipedia corpus (Step 2)


def load_hotpot(n: int, seed: int = 42, split: str = "validation") -> Tuple[List[dict], List[dict]]:
    """Returns (eval records, pool). Pool = every paragraph in the split (closed pool: easier than open-domain)."""
    ds = _load([("hotpotqa/hotpot_qa", "distractor"), ("hotpot_qa", "distractor")], split=split)
    pool: Dict[str, dict] = {}
    for row in ds:
        for t, text in hotpot_paragraphs(row):
            pool.setdefault(text, {"id": t, "text": text})
    recs = [hotpot_record(ds[i]) for i in _sample(range(len(ds)), n, seed)]
    return recs, list(pool.values())


def load_pubmed(n: int, seed: int = 42, n_train: int = 0) -> Tuple[List[dict], List[dict], List[dict]]:
    """pqa_labeled has 1000 rows. Returns (eval, pool, train): eval = first n of a seeded shuffle,
    train = the next n_train (kept disjoint from eval for Step 5 fine-tuning)."""
    ds = _load([("qiaojin/PubMedQA", "pqa_labeled"), ("pubmed_qa", "pqa_labeled")], split="train")
    order = _sample(range(len(ds)), None, seed)
    allrecs = [pubmed_record(ds[i]) for i in order]
    pool = [{"id": r["gold_ids"][0], "text": r["gold_passages"][0]} for r in allrecs]
    return allrecs[:n], pool, allrecs[n : n + n_train]


def build_pool(dev_rows_paragraphs, extra_paragraphs, pool_size: int) -> List[dict]:
    """Pure helper. Pool = every dev paragraph (so every gold passage is present) + extra paragraphs
    (other splits) until the pool has `pool_size` unique texts. pool_size <= len(dev) keeps dev only."""
    pool: Dict[str, dict] = {}
    for t, text in dev_rows_paragraphs:
        pool.setdefault(text, {"id": t, "text": text})
    for t, text in extra_paragraphs:
        if len(pool) >= pool_size:
            break
        pool.setdefault(text, {"id": t, "text": text})
    return list(pool.values())


def load_hotpot_open(n: int, seed: int = 42, pool_size: int = 0) -> Tuple[List[dict], List[dict]]:
    """HotpotQA dev questions with a LARGER pool: all dev paragraphs + HotpotQA-train paragraphs up to
    `pool_size` unique passages (0 = dev paragraphs only, as in Step 1). n=0 means the full dev set (7,405).
    The eval subset is a prefix of one seeded shuffle, so n=2000 is contained in n=7405."""
    ds = _load([("hotpotqa/hotpot_qa", "distractor"), ("hotpot_qa", "distractor")], split="validation")
    dev = [tp for row in ds for tp in hotpot_paragraphs(row)]
    extra = []
    if pool_size:
        tr = _load([("hotpotqa/hotpot_qa", "distractor"), ("hotpot_qa", "distractor")], split="train")
        for i in _sample(range(len(tr)), None, seed):
            extra.extend(hotpot_paragraphs(tr[i]))
            if len(extra) > 12 * pool_size // 10 + 200:  # enough to reach pool_size after de-duplication
                break
    recs = [hotpot_record(ds[i]) for i in _sample(range(len(ds)), n or None, seed)]
    return recs, build_pool(dev, extra, pool_size)


def load_pubmed_open(n: int, seed: int = 42, pool_size: int = 0, n_train: int = 0):
    """PubMedQA eval from pqa_labeled (1,000 expert-labelled) with a LARGER abstract pool:
    the 1,000 labelled abstracts + pqa_artificial abstracts up to `pool_size`."""
    recs, pool, train = load_pubmed(n, seed, n_train)
    if pool_size and pool_size > len(pool):
        art = _load([("qiaojin/PubMedQA", "pqa_artificial"), ("pubmed_qa", "pqa_artificial")], split="train")
        seen = {p["text"] for p in pool}
        for i in _sample(range(len(art)), None, seed):
            if len(pool) >= pool_size:
                break
            txt = " ".join(art[i]["context"]["contexts"]).strip()
            if txt and txt not in seen:
                seen.add(txt)
                pool.append({"id": str(art[i]["pubid"]), "text": txt})
    return recs, pool, train


def load_nq_beir(n: int, seed: int = 42, pool_size: int = 200_000):
    """OPTIONAL open-domain NQ. NQ-open questions (short answers) matched by question text to the BEIR/MTEB NQ
    test queries, whose relevance labels point at Wikipedia passages in a 2.68M-passage corpus. Pool = the gold
    passages of the matched questions + a seeded random sample of the corpus up to `pool_size`.
    Run `python slmrag_data.py --check_nq` first: the schema could not be verified offline."""
    def norm(q):
        return " ".join(q.lower().replace("?", " ").split())

    nq = _load([("google-research-datasets/nq_open",), ("nq_open",)], split="validation")
    ans = {norm(nq[i]["question"]): list(nq[i]["answer"]) for i in range(len(nq))}
    corpus = _load([("mteb/nq", "corpus")], split="corpus")
    queries = _load([("mteb/nq", "queries")], split="queries")
    qrels = _load([("mteb/nq", "default")], split="test")
    qtext = {r["_id"]: r["text"] for r in queries}
    gold: Dict[str, List[str]] = {}
    for r in qrels:
        if r.get("score", 1) > 0:
            gold.setdefault(r["query-id"], []).append(r["corpus-id"])
    cid_to_row = None  # built lazily only for gold ids
    wanted = {c for v in gold.values() for c in v}
    ids = corpus["_id"]
    cid_to_row = {c: i for i, c in enumerate(ids) if c in wanted}
    def ptext(i):
        return f"{corpus[i]['title']}: {corpus[i]['text']}".strip()
    recs, gold_rows = [], set()
    for qid, cids in gold.items():
        a = ans.get(norm(qtext.get(qid, "")))
        rows = [cid_to_row[c] for c in cids if c in cid_to_row]
        if a and rows:
            recs.append({"id": f"nq-{qid}", "question": qtext[qid].strip().rstrip("?") + "?", "task": "qa",
                         "answers": a, "gold_passages": [ptext(i) for i in rows],
                         "gold_ids": [str(corpus[i]["_id"]) for i in rows]})
            gold_rows.update(rows)
    random.Random(seed).shuffle(recs)
    recs = recs[:n] if n else recs
    keep = {i for r in recs for i in (cid_to_row[g] for g in r["gold_ids"])}
    pool = {i: {"id": str(ids[i]), "text": ptext(i)} for i in keep}
    for i in _sample(range(len(ids)), None, seed):
        if len(pool) >= pool_size:
            break
        pool.setdefault(i, {"id": str(ids[i]), "text": ptext(i)})
    return recs, list(pool.values())


def load_corpus_jsonl(path: str) -> List[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


if __name__ == "__main__":
    if "--check_nq" in sys.argv:
        r, p = load_nq_beir(5, pool_size=1000)
        print(len(r), "questions;", len(p), "pool passages;", {k: (v if k != "gold_passages" else [x[:80] for x in v]) for k, v in r[0].items()})
        assert r and r[0]["gold_passages"], "NQ-BEIR: no matched questions with gold passages"
        print("NQ-BEIR schema OK.")
    if "--check" in sys.argv:
        nq, _ = load_nq(2)
        hp, pool = load_hotpot(2)
        pm, ppool, _ = load_pubmed(2)
        for name, r in (("nq", nq[0]), ("hotpot", hp[0]), ("pubmed", pm[0])):
            print(name, {k: (v if k != "gold_passages" else [p[:80] for p in v]) for k, v in r.items()})
        assert hp[0]["gold_passages"], "HotpotQA: supporting-fact titles did not match context titles"
        assert pm[0]["label"] in ("yes", "no", "maybe")
        print(f"hotpot pool={len(pool)} paragraphs, pubmed pool={len(ppool)} abstracts. Schemas OK.")
        art = _load([("qiaojin/PubMedQA", "pqa_artificial"), ("pubmed_qa", "pqa_artificial")], split="train[:2]")
        assert " ".join(art[0]["context"]["contexts"]).strip(), "pqa_artificial schema differs"
        print("pqa_artificial schema OK.")

Overwriting slmrag_data.py


In [5]:
%%writefile slmrag_hf.py
"""Hugging Face generation backend for Step 1 (needs torch + transformers; not unit-tested
in the dev sandbox, which has no GPU or model access -- see PROTOCOL.md smoke test).

Key differences from the old pipeline that caused the F1 floor:
  * chat template + a short-answer system prompt
  * generation stops on the model's real end-of-turn tokens (not just the base EOS)
  * max_new_tokens=32, greedy
  * the RAW generation and a hit-the-limit flag are returned so rambling is measurable
"""
from __future__ import annotations

from typing import List, Tuple


def load_model(name: str, load_4bit: bool = False):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = {}
    if load_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    else:
        # T4 has no bf16. Gemma-2 can overflow in fp16: use load_4bit or fp32 for it.
        kw["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(name, device_map="auto", **kw).eval()
    return tok, model


def eos_ids(tok, model) -> List[int]:
    ids = set()
    g = getattr(model.generation_config, "eos_token_id", None)
    if isinstance(g, int):
        ids.add(g)
    elif g:
        ids.update(int(x) for x in g)
    if tok.eos_token_id is not None:
        ids.add(int(tok.eos_token_id))
    for t in ("<|im_end|>", "<|eot_id|>", "<end_of_turn>", "<|end|>"):
        i = tok.convert_tokens_to_ids(t)
        if isinstance(i, int) and i >= 0 and i != tok.unk_token_id:
            ids.add(i)
    return sorted(ids)


def render(tok, messages: List[dict]) -> str:
    """Apply the chat template; fold the system prompt into the user turn for templates that reject it (Gemma)."""
    try:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        merged = [{"role": "user", "content": messages[0]["content"] + "\n\n" + messages[1]["content"]}]
        return tok.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)


def make_generate_fn(tok, model, max_new_tokens: int = 32, batch_size: int = 16):
    import torch

    stops = set(eos_ids(tok, model))

    def generate(prompts: List[List[dict]]) -> List[Tuple[str, bool]]:
        texts = [render(tok, m) for m in prompts]
        lens = [len(tok(t, add_special_tokens=False).input_ids) for t in texts]
        order = sorted(range(len(texts)), key=lambda i: lens[i])  # length-sorted batches: less padding
        results: List[Tuple[str, bool]] = [("", False)] * len(texts)
        i, bs = 0, batch_size
        while i < len(order):
            idx = order[i : i + bs]
            enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
            try:
                with torch.no_grad():
                    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                         eos_token_id=sorted(stops), pad_token_id=tok.pad_token_id)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)  # retry the same slice with a smaller batch
                continue
            new = out[:, enc.input_ids.shape[1]:]
            for row, j in zip(new, idx):
                ids = row.tolist()
                hit_eos = any(t in stops for t in ids)
                results[j] = (tok.decode(ids, skip_special_tokens=True), not hit_eos)
            i += len(idx)
        return results

    return generate


def make_label_fn(tok, model, batch_size: int = 8):
    """Score yes/no/maybe by the next-token distribution right after the chat prompt (no free text, no parsing).
    Each label's probability = sum over its lower/Capitalised first-token variants, renormalised over the 3 labels."""
    import torch

    variants = {}
    for lab in ("yes", "no", "maybe"):
        ids = {tok.encode(v, add_special_tokens=False)[0] for v in (lab, lab.capitalize(), " " + lab, " " + lab.capitalize())}
        variants[lab] = sorted(ids)
    allids = [i for v in variants.values() for i in v]
    assert len(allids) == len(set(allids)), f"yes/no/maybe first tokens overlap for this tokenizer: {variants}"

    def score(prompts: List[List[dict]]):
        texts = [render(tok, m) for m in prompts]
        order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
        res = [None] * len(texts)
        i, bs = 0, batch_size
        while i < len(order):
            idx = order[i : i + bs]
            enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            try:
                with torch.no_grad():
                    try:
                        lg = model(**enc, logits_to_keep=1).logits[:, -1, :]
                    except TypeError:
                        lg = model(**enc).logits[:, -1, :]
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                continue
            lp = torch.log_softmax(lg.float(), dim=-1)
            for row, j in zip(lp, idx):
                m = {lab: torch.logsumexp(row[v], dim=0) for lab, v in variants.items()}
                p = torch.softmax(torch.stack([m["yes"], m["no"], m["maybe"]]), dim=0).tolist()
                res[j] = {"yes": p[0], "no": p[1], "maybe": p[2]}
            i += len(idx)
        return res

    return score

Overwriting slmrag_hf.py


In [6]:
%%writefile run_step2.py
"""Step 2 runner: bigger retrieval pool, controls, PubMedQA logit scoring, resumable.

  python run_step2.py --dataset hotpot --model Qwen/Qwen2.5-1.5B-Instruct --n 0 --pool_size 200000   # full dev set
  python run_step2.py --dataset hotpot --model Qwen/Qwen2.5-0.5B-Instruct --n 2000 --pool_size 200000 \
         --settings closed,oracle,oracle_noise,random,bm25,dense,hybrid
  python run_step2.py --dataset pubmed --model Qwen/Qwen2.5-1.5B-Instruct --n 1000 --pool_size 50000 --k 3
  python run_step2.py --dataset hotpot --retrieval_only --n 0 --pool_size 200000      # no LLM at all

Settings
  closed        no context
  oracle        gold passages only
  oracle_noise  gold passages hidden among random passages (same k, shuffled order)  -> utilisation under noise
  random        k random passages (no gold)                                          -> pure distraction control
  bm25 | dense | hybrid   real retrieval over the pool (top-k)

Retrieval is model-independent, so it is computed once and cached (embeddings + top-10 lists) and reused by
every model. Each (setting) result is written as soon as it finishes; --resume skips finished ones, so a
crashed Kaggle session loses at most one setting.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import random
import re
import sys
import time

import numpy as np

import slmrag_core as C
import slmrag_data as D

TOP = 10


def slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "-", s).strip("-")


def pool_fingerprint(pool) -> str:
    h = hashlib.md5()
    h.update(str(len(pool)).encode())
    for p in pool[:: max(1, len(pool) // 997)]:
        h.update(p["text"][:200].encode())
    return h.hexdigest()[:10]


def compute_retrieval(pool, recs, dense_model, cache_dir, tag, encode_fn=None, depth=100, need=("bm25", "dense", "hybrid"), top=TOP):
    """-> {setting: {question id: [pool indices, best first, length TOP]}}. Cached on disk under cache_dir."""
    os.makedirs(cache_dir, exist_ok=True)
    fp = pool_fingerprint(pool)
    qh = hashlib.md5("|".join(r["id"] for r in recs).encode()).hexdigest()[:8]
    rpath = os.path.join(cache_dir, f"retr_{tag}_{fp}_{qh}_{slug(dense_model)}_top{top}.json")
    if os.path.exists(rpath):
        print("retrieval cache hit:", rpath, flush=True)
        return json.load(open(rpath))
    texts = [p["text"] for p in pool]
    qs = [r["question"] for r in recs]
    t0 = time.time()
    print(f"BM25 index over {len(texts):,} passages ...", flush=True)
    bm = C.BM25(texts)
    bm_lists = [[i for i, _ in bm.search(q, depth)] for q in qs]
    print(f"  bm25 done in {time.time() - t0:.0f}s", flush=True)

    epath = os.path.join(cache_dir, f"emb_{tag}_{fp}_{slug(dense_model)}.npy")
    qp, dp = ("query: ", "passage: ") if "e5" in dense_model.lower() else ("", "")
    enc = encode_fn or C.make_st_encoder(dense_model)
    emb = np.load(epath) if os.path.exists(epath) else None
    t0 = time.time()
    dr = C.DenseRetriever(texts, enc, query_prefix=qp, doc_prefix=dp, embeddings=emb)
    if emb is None:
        np.save(epath, dr.E)
    print(f"  dense embeddings ready in {time.time() - t0:.0f}s", flush=True)
    de_lists = [[i for i, _ in res] for res in dr.search_batch(qs, depth)]
    hy_lists = [[i for i, _ in C.rrf([a, b], k=60, top=top)] for a, b in zip(bm_lists, de_lists)]
    out = {"bm25": {r["id"]: l[:top] for r, l in zip(recs, bm_lists)},
           "dense": {r["id"]: l[:top] for r, l in zip(recs, de_lists)},
           "hybrid": {r["id"]: l[:top] for r, l in zip(recs, hy_lists)}}
    json.dump(out, open(rpath, "w"))
    return out


def retrieval_report(retr, pool, recs, ks=(1, 5, 10)):
    """Answer-containment + gold-document metrics per retriever. 'nonyn' drops HotpotQA yes/no questions."""
    rep = {}
    nonyn = [r for r in recs if not C.is_yesno_gold(r["answers"])] if recs and recs[0]["task"] == "qa" else recs
    for name, per_q in retr.items():
        m = {}
        for tag, rs in (("", recs), ("nonyn_", nonyn)):
            if tag and len(rs) == len(recs):
                continue
            if rs and rs[0]["task"] == "qa":
                texts = [[pool[i]["text"] for i in per_q[r["id"]]] for r in rs]
                m.update({tag + k: v for k, v in C.retrieval_metrics(texts, [r["answers"] for r in rs], ks).items()})
            ids = [[pool[i]["id"] for i in per_q[r["id"]]] for r in rs]
            gold = [r["gold_ids"] for r in rs]
            for k in ks:
                m[f"{tag}id_recall@{k}"] = C.id_recall_at_k(ids, gold, k)
                m[f"{tag}all_gold@{k}"] = C.all_gold_at_k(ids, gold, k)
        rep[name] = m
    return rep


def make_context_cache(setting, recs, pool, retr, k, seed):
    """-> {question id: [passage strings]} for a setting, or None for closed-book."""
    if setting == "closed":
        return None
    if setting == "oracle":
        return {r["id"]: list(r["gold_passages"]) for r in recs}
    if setting in ("bm25", "dense", "hybrid"):
        return {r["id"]: [pool[i]["text"] for i in retr[setting][r["id"]][:k]] for r in recs}
    out = {}
    for r in recs:
        rng = random.Random(f"{seed}-{setting}-{r['id']}")
        gold = set(r["gold_passages"])
        fill = []
        while len(fill) < (k if setting == "random" else max(0, k - len(gold))):
            c = pool[rng.randrange(len(pool))]["text"]
            if c not in gold and c not in fill:
                fill.append(c)
        ctx = fill if setting == "random" else list(r["gold_passages"]) + fill
        rng.shuffle(ctx)
        out[r["id"]] = ctx
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", choices=["hotpot", "pubmed", "nq"], required=True)
    ap.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    ap.add_argument("--settings", default="closed,oracle,bm25,dense,hybrid")
    ap.add_argument("--n", type=int, default=0, help="0 = all available questions (HotpotQA dev: 7,405)")
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--pool_size", type=int, default=0, help="unique passages in the retrieval pool (0 = dataset's own)")
    ap.add_argument("--max_new_tokens", type=int, default=32)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--context_budget", type=int, default=600)
    ap.add_argument("--dense_model", default="intfloat/e5-base-v2")
    ap.add_argument("--yn_mode", choices=["logit", "generate"], default="logit", help="PubMedQA scoring")
    ap.add_argument("--load_4bit", action="store_true")
    ap.add_argument("--retrieval_only", action="store_true")
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--cache_dir", default="cache")
    ap.add_argument("--out", default="runs2")
    args = ap.parse_args(argv)

    settings = args.settings.split(",")
    if args.smoke:
        args.n, settings, args.pool_size = 20, ["closed", "oracle", "oracle_noise", "random"], 0
    os.makedirs(args.out, exist_ok=True)

    if args.dataset == "hotpot":
        recs, pool = D.load_hotpot_open(args.n, args.seed, args.pool_size)
    elif args.dataset == "pubmed":
        recs, pool, _ = D.load_pubmed_open(args.n or 1000, args.seed, args.pool_size)
    else:
        recs, pool = D.load_nq_beir(args.n, args.seed, args.pool_size or 200_000)
    print(f"{args.dataset}: {len(recs)} questions, pool {len(pool):,} passages", flush=True)
    ptag = f"{args.dataset}_p{len(pool)}"

    retr, rep = {}, {}
    need = {"bm25", "dense", "hybrid"} & set(settings)
    if need or args.retrieval_only:
        retr = compute_retrieval(pool, recs, args.dense_model, args.cache_dir, ptag)
        rep = retrieval_report(retr, pool, recs)
        for name, m in rep.items():
            print(f"  {name:7s} " + "  ".join(f"{k}={v:.3f}" for k, v in m.items() if "@" in k and not k.startswith("nonyn")), flush=True)
        with open(os.path.join(args.out, f"retrieval__{ptag}__n{len(recs)}.json"), "w") as f:
            json.dump({"pool": len(pool), "n": len(recs), "metrics": rep}, f, indent=1)
    if args.retrieval_only:
        return

    from slmrag_hf import load_model, make_generate_fn, make_label_fn
    tok, model = load_model(args.model, args.load_4bit)
    gen = make_generate_fn(tok, model, args.max_new_tokens, args.batch_size)
    label_fn = make_label_fn(tok, model) if (args.dataset == "pubmed" and args.yn_mode == "logit") else None

    summaries = {}
    stem = f"{args.dataset}__{slug(args.model)}"
    for s in settings:
        base = os.path.join(args.out, f"{stem}__{s}_k{args.k}_n{len(recs)}_p{len(pool)}")
        if args.resume and os.path.exists(base + ".jsonl") and os.path.exists(base + ".summary.json"):
            summaries[s] = json.load(open(base + ".summary.json"))
            print(f"{s}: resumed from disk", flush=True)
            continue
        if s == "oracle" and all(not r["gold_passages"] for r in recs):
            print("oracle skipped: no gold passages", file=sys.stderr)
            continue
        cache = make_context_cache(s, recs, pool, retr, args.k, args.seed)
        retrieve_fn = (lambda r, c=cache: c[r["id"]]) if cache is not None else None
        t0 = time.time()
        rows, summ = C.evaluate(recs, gen, retrieve_fn, context_budget=args.context_budget, label_fn=label_fn)
        summ["seconds"] = time.time() - t0
        if s in rep:
            summ.update({f"ret_{k}": v for k, v in rep[s].items()})
        summaries[s] = summ
        with open(base + ".jsonl", "w") as f:
            for row in rows:
                f.write(json.dumps(row) + "\n")
        with open(base + ".summary.json", "w") as f:
            json.dump({"args": vars(args), "setting": s, "pool": len(pool), **summ}, f, indent=1)
        key = "yn_accuracy" if args.dataset == "pubmed" else "acc"
        extra = (f" cal_acc={summ['yn_cal_accuracy']:.3f} majority={summ['yn_majority_acc']:.3f}"
                 if "yn_cal_accuracy" in summ else f" f1={summ.get('f1', float('nan')):.3f}")
        print(f"{s:12s} n={summ['n']} {key}={summ.get(key, float('nan')):.3f}{extra} "
              f"words={summ.get('mean_pred_words', float('nan')):.1f} hit_max={summ['hit_max_rate']:.1%} "
              f"({summ['seconds'] / 60:.1f} min)", flush=True)

    for w in C.sanity_warnings({k: v for k, v in summaries.items() if k in ("closed", "oracle")}):
        print("WARNING:", w)
    with open(os.path.join(args.out, f"{stem}__ALL_n{len(recs)}_p{len(pool)}.json"), "w") as f:
        json.dump(summaries, f, indent=1)


if __name__ == "__main__":
    main()

Overwriting run_step2.py


In [7]:
%%writefile run_step3.py
"""Step 3 runner: WHY does the small model fail to read evidence it has?

Reuses the Step 2 data, retrieval and caching code and adds
  * --variant base|own|cot       system prompt (own = 'use your own knowledge if the context does not help';
                                 cot = three short sentences of reasoning, then 'Final answer: ...')
  * order/position settings      oracle_rev, oracle_noise_first, oracle_noise_last
  * file names that carry k, budget and variant so sweeps never overwrite each other

  python run_step3.py --dataset hotpot --model Qwen/Qwen2.5-1.5B-Instruct --n 2000 --pool_size 200000 \
      --k 10 --context_budget 4000 --settings dense --resume
  python run_step3.py ... --variant cot --settings oracle,dense,random,closed

Settings: closed oracle oracle_rev oracle_noise oracle_noise_first oracle_noise_last random bm25 dense hybrid
"""
from __future__ import annotations

import argparse
import json
import os
import random
import sys
import time

import slmrag_core as C
import slmrag_data as D
import run_step2 as R2

NEW = ("oracle_rev", "oracle_noise_first", "oracle_noise_last")
ALL_SETTINGS = ("closed", "oracle", "oracle_rev", "oracle_noise", "oracle_noise_first", "oracle_noise_last",
                "random", "bm25", "dense", "hybrid")


def make_context_cache(setting, recs, pool, retr, k, seed):
    if setting not in NEW:
        return R2.make_context_cache(setting, recs, pool, retr, k, seed)
    out = {}
    for r in recs:
        gold = list(r["gold_passages"])
        if setting == "oracle_rev":
            out[r["id"]] = gold[::-1]
            continue
        rng = random.Random(f"{seed}-{setting}-{r['id']}")
        fill = []
        while len(fill) < max(0, k - len(gold)):
            c = pool[rng.randrange(len(pool))]["text"]
            if c not in gold and c not in fill:
                fill.append(c)
        out[r["id"]] = gold + fill if setting == "oracle_noise_first" else fill + gold
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", choices=["hotpot", "nq"], default="hotpot")
    ap.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    ap.add_argument("--settings", default="closed,oracle,dense,random")
    ap.add_argument("--variant", choices=["base", "own", "cot"], default="base")
    ap.add_argument("--n", type=int, default=2000)
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--pool_size", type=int, default=200000)
    ap.add_argument("--max_new_tokens", type=int, default=0, help="0 = 32 (160 for --variant cot)")
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--context_budget", type=int, default=2000, help="words; large so no passage is cut away")
    ap.add_argument("--dense_model", default="intfloat/e5-base-v2")
    ap.add_argument("--load_4bit", action="store_true")
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--cache_dir", default="cache")
    ap.add_argument("--out", default="runs3")
    args = ap.parse_args(argv)
    if not args.max_new_tokens:
        args.max_new_tokens = 160 if args.variant == "cot" else 32

    settings = args.settings.split(",")
    bad = [s for s in settings if s not in ALL_SETTINGS]
    if bad:
        raise SystemExit(f"unknown settings {bad}; choose from {ALL_SETTINGS}")
    if args.smoke:
        args.n, settings, args.pool_size = 20, ["closed", "oracle", "oracle_rev", "oracle_noise_last"], 0
    os.makedirs(args.out, exist_ok=True)

    if args.dataset == "hotpot":
        recs, pool = D.load_hotpot_open(args.n, args.seed, args.pool_size)
    else:
        recs, pool = D.load_nq_beir(args.n, args.seed, args.pool_size or 200_000)
    print(f"{args.dataset}: {len(recs)} questions, pool {len(pool):,} passages, variant={args.variant}, k={args.k}", flush=True)
    ptag = f"{args.dataset}_p{len(pool)}"

    retr, rep = {}, {}
    if {"bm25", "dense", "hybrid"} & set(settings):
        top = max(20, args.k)   # one cache serves every k up to 20
        retr = R2.compute_retrieval(pool, recs, args.dense_model, args.cache_dir, ptag, top=top)
        rep = R2.retrieval_report(retr, pool, recs, ks=tuple(sorted({1, 5, 10, args.k})))

    from slmrag_hf import load_model, make_generate_fn
    tok, model = load_model(args.model, args.load_4bit)
    gen = make_generate_fn(tok, model, args.max_new_tokens, args.batch_size)

    stem = f"{args.dataset}__{R2.slug(args.model)}"
    summaries = {}
    for s in settings:
        base = os.path.join(args.out, f"{stem}__{s}_k{args.k}_n{len(recs)}_p{len(pool)}_v{args.variant}_b{args.context_budget}")
        if args.resume and os.path.exists(base + ".jsonl") and os.path.exists(base + ".summary.json"):
            summaries[s] = json.load(open(base + ".summary.json"))
            print(f"{s}: resumed from disk", flush=True)
            continue
        cache = make_context_cache(s, recs, pool, retr, args.k, args.seed)
        retrieve_fn = (lambda r, c=cache: c[r["id"]]) if cache is not None else None
        t0 = time.time()
        rows, summ = C.evaluate(recs, gen, retrieve_fn, context_budget=args.context_budget, variant=args.variant)
        summ["seconds"] = time.time() - t0
        if s in rep:
            summ.update({f"ret_{k}": v for k, v in rep[s].items()})
        summaries[s] = summ
        with open(base + ".jsonl", "w") as f:
            for row in rows:
                f.write(json.dumps(row) + "\n")
        with open(base + ".summary.json", "w") as f:
            json.dump({"args": vars(args), "setting": s, "pool": len(pool), **summ}, f, indent=1)
        print(f"{s:19s} n={summ['n']} acc_nonyn={summ.get('acc_nonyn', summ.get('acc', float('nan'))):.3f} "
              f"em={summ.get('em', float('nan')):.3f} words={summ.get('mean_pred_words', float('nan')):.1f} "
              f"final_found={summ['final_found_rate']:.0%} hit_max={summ['hit_max_rate']:.1%} ({summ['seconds'] / 60:.1f} min)", flush=True)

    for w in C.sanity_warnings({k: v for k, v in summaries.items() if k in ("closed", "oracle")}):
        print("WARNING:", w)
    with open(os.path.join(args.out, f"{stem}__ALL_k{args.k}_n{len(recs)}_p{len(pool)}_v{args.variant}_b{args.context_budget}.json"), "w") as f:
        json.dump(summaries, f, indent=1)


if __name__ == "__main__":
    main()

Overwriting run_step3.py


In [8]:
%%writefile slmrag_analysis.py
"""Analysis helpers for Step 2 (pure functions; unit-tested).

load_rows        read a run's per-example JSONL into {id: row}
distraction_rescue   how often context destroys a correct closed-book answer / fixes a wrong one
utilization      accuracy when the gold string IS in the context vs not (the 'found it but did not use it' gap)
gain_table       paired gain of each setting over closed-book (CI, McNemar/Wilcoxon, Holm)
"""
from __future__ import annotations

import json
from typing import Dict, List, Optional, Sequence

import numpy as np

import slmrag_core as C


def load_rows(path: str) -> Dict[str, dict]:
    with open(path) as f:
        return {r["id"]: r for r in map(json.loads, f)}


def _ids(base: Dict[str, dict], cur: Dict[str, dict], nonyn: bool) -> List[str]:
    ids = sorted(set(base) & set(cur))
    if nonyn:
        ids = [i for i in ids if not base[i].get("yn_q")]
    return ids


def distraction_rescue(base: Dict[str, dict], cur: Dict[str, dict], nonyn: bool = True) -> dict:
    """base = closed-book rows, cur = rows with context. Binary 'acc' (containment) is used.
    lost    = P(wrong with context | right closed-book)      (distraction)
    rescued = P(right with context | wrong closed-book)      (what context adds)
    Both are conditional on the closed-book outcome, so regression to the mean inflates `lost`
    a little: compare `lost` of real retrieval with `lost` of the `random` control, not with 0."""
    ids = _ids(base, cur, nonyn)
    b = np.array([base[i]["acc"] for i in ids]) > 0.5
    c = np.array([cur[i]["acc"] for i in ids]) > 0.5
    out = {"n": len(ids), "n_base_right": int(b.sum()), "n_base_wrong": int((~b).sum())}
    if b.sum():
        m, lo, hi = C.bootstrap_ci((~c[b]).astype(float), B=2000)
        out.update(lost=m, lost_lo=lo, lost_hi=hi)
    if (~b).sum():
        m, lo, hi = C.bootstrap_ci(c[~b].astype(float), B=2000)
        out.update(rescued=m, rescued_lo=lo, rescued_hi=hi)
    return out


def utilization(rows: Dict[str, dict], nonyn: bool = True) -> dict:
    rs = [r for r in rows.values() if "ctx_has_answer" in r and not (nonyn and r.get("yn_q"))]
    has = [r["acc"] for r in rs if r["ctx_has_answer"]]
    no = [r["acc"] for r in rs if not r["ctx_has_answer"]]
    return {"n_ctx_has_answer": len(has), "acc_given_answer_in_ctx": float(np.mean(has)) if has else float("nan"),
            "n_ctx_lacks_answer": len(no), "acc_given_no_answer_in_ctx": float(np.mean(no)) if no else float("nan"),
            "share_ctx_has_answer": len(has) / max(1, len(rs))}


def gain_table(base: Dict[str, dict], others: Dict[str, Dict[str, dict]], metric: str = "acc", nonyn: bool = True) -> List[dict]:
    """Paired comparison of every setting in `others` against `base`, Holm-corrected over the family."""
    res = []
    for name, cur in others.items():
        ids = _ids(base, cur, nonyn)
        a, b = [cur[i][metric] for i in ids], [base[i][metric] for i in ids]
        r = C.paired_compare(a, b, binary=metric != "f1", B=4000)
        r["setting"] = name
        r["p"] = r.get("p_mcnemar", r["p_wilcoxon"])
        res.append(r)
    adj = C.holm([r["p"] for r in res]) if res else []
    for r, p in zip(res, adj):
        r["p_holm"] = p
        r["mde80"] = C.mde_paired(r["n"], r["sd_diff"]) if r["sd_diff"] > 0 else float("nan")
    return res


def compare_pair(a: Dict[str, dict], b: Dict[str, dict], metric: str = "acc", nonyn: bool = True) -> dict:
    ids = _ids(a, b, nonyn)
    r = C.paired_compare([b[i][metric] for i in ids], [a[i][metric] for i in ids], binary=metric != "f1", B=4000)
    r["mde80"] = C.mde_paired(r["n"], r["sd_diff"]) if r["sd_diff"] > 0 else float("nan")
    return r


# ---- Step 3 helpers ---------------------------------------------------------
import re as _re

ABSTAIN = _re.compile(r"(not (mentioned|specified|provided|stated|given|available|clear|applicable)|unknown|no (information|answer|such)|"
                      r"cannot|can't|unable|insufficient|n/a|\bnone\b|unclear|does not|neither)", _re.I)


def error_taxonomy(rows: Dict[str, dict], nonyn: bool = True) -> Dict[str, float]:
    """Where do answers go wrong? correct | wrong_copied (wrong but the answer text IS somewhere in the context:
    the model picked a different span/entity) | abstain (says unknown/not applicable) | wrong_other | empty."""
    rs = [r for r in rows.values() if r.get("task", "qa") == "qa" and not (nonyn and r.get("yn_q"))]
    n = max(1, len(rs))
    cat = {"correct": 0, "wrong_copied": 0, "abstain": 0, "wrong_other": 0, "empty": 0}
    for r in rs:
        if r["acc"] > 0.5:
            cat["correct"] += 1
        elif not r["pred"]:
            cat["empty"] += 1
        elif ABSTAIN.search(r["pred"]):
            cat["abstain"] += 1
        elif r.get("pred_in_ctx"):
            cat["wrong_copied"] += 1
        else:
            cat["wrong_other"] += 1
    return {k: v / n for k, v in cat.items()} | {"n": len(rs)}


def group_acc(rows: Dict[str, dict], key: str, nonyn: bool = True) -> Dict[str, Tuple[float, int]]:
    out: Dict[str, List[float]] = {}
    for r in rows.values():
        if r.get("task", "qa") != "qa" or (nonyn and r.get("yn_q")) or r.get(key) is None:
            continue
        out.setdefault(str(r[key]), []).append(r["acc"])
    return {g: (float(np.mean(v)), len(v)) for g, v in sorted(out.items())}

Overwriting slmrag_analysis.py


In [9]:
%%writefile analyze_step2.py
"""Read a runs2/ folder and print the Step 2 tables (also saves CSVs).

  python analyze_step2.py runs2 --dataset hotpot --pool 200000 --n 2000

Prints, per model: accuracy / EM / F1 (all questions and non-yes/no), paired gain over closed-book (Holm),
distraction vs rescue (with the random-context control), utilisation gap, retriever head-to-head,
and the model-size trend. Everything is aligned by question id.
"""
from __future__ import annotations

import argparse
import glob
import os
import re

import pandas as pd

import slmrag_analysis as A

SETTINGS = ["closed", "oracle", "oracle_noise", "random", "bm25", "dense", "hybrid"]


def find_runs(folder, dataset, n, pool, k):
    runs = {}
    for f in glob.glob(os.path.join(folder, f"{dataset}__*__*_k{k}_n{n}_p{pool}.jsonl")):
        m = re.match(rf"{dataset}__(.+)__({'|'.join(SETTINGS)})_k{k}_n{n}_p{pool}\.jsonl$", os.path.basename(f))
        if m:
            runs.setdefault(m.group(1), {})[m.group(2)] = A.load_rows(f)
    return runs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folder")
    ap.add_argument("--dataset", default="hotpot")
    ap.add_argument("--n", type=int, default=None, help="number of questions in the run (see file names); auto if omitted")
    ap.add_argument("--pool", type=int, default=None, help="pool size in the file names; auto if omitted")
    ap.add_argument("--k", type=int, default=5)
    a = ap.parse_args()
    if a.n is None or a.pool is None:
        combos = sorted({(int(m.group(1)), int(m.group(2))) for f in glob.glob(os.path.join(a.folder, f"{a.dataset}__*_k{a.k}_n*_p*.jsonl"))
                         for m in [re.search(r"_n(\d+)_p(\d+)\.jsonl$", f)] if m})
        if len(combos) != 1:
            raise SystemExit(f"pass --n and --pool; found (n, pool) combinations: {combos}")
        a.n, a.pool = combos[0]
        print(f"auto-detected n={a.n} pool={a.pool}")
    runs = find_runs(a.folder, a.dataset, a.n, a.pool, a.k)
    if not runs:
        raise SystemExit("no matching runs: check --n / --pool against the file names in the folder")
    table = []
    for model, R in sorted(runs.items()):
        print("=" * 100, f"\nMODEL {model}   settings: {sorted(R)}")
        base = R.get("closed")
        for s in [x for x in SETTINGS if x in R]:
            rows = list(R[s].values())
            qa = [r for r in rows if not r.get("yn_q")]
            m = lambda key, rs: sum(r[key] for r in rs) / max(1, len(rs))
            table.append({"model": model, "setting": s, "n": len(rows), "acc": m("acc", rows), "acc_nonyn": m("acc", qa),
                          "em_nonyn": m("em", qa), "f1_nonyn": m("f1", qa),
                          **{f"util_{k}": v for k, v in A.utilization(R[s]).items() if k != "n_ctx_lacks_answer"}})
        if base:
            others = {s: R[s] for s in SETTINGS[1:] if s in R}
            print("\n-- gain over closed-book (non-yes/no, containment acc), Holm over this family --")
            for r in A.gain_table(base, others):
                print(f"  {r['setting']:13s} n={r['n']} diff={r['mean_diff']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] "
                      f"p_holm={r['p_holm']:.3g}  MDE80={r['mde80']:.3f}")
            print("\n-- distraction (lost) vs rescue; compare 'random' with the real retrievers --")
            for s in others:
                d = A.distraction_rescue(base, R[s])
                print(f"  {s:13s} lost {d.get('lost', float('nan')):.1%} [{d.get('lost_lo', float('nan')):.1%},{d.get('lost_hi', float('nan')):.1%}]"
                      f"  rescued {d.get('rescued', float('nan')):.1%}   (closed right: {d['n_base_right']}, wrong: {d['n_base_wrong']})")
        print("\n-- utilisation: accuracy when the gold string is / is not in the prompt context --")
        for s in [x for x in SETTINGS[1:] if x in R]:
            u = A.utilization(R[s])
            print(f"  {s:13s} in-context {u['acc_given_answer_in_ctx']:.3f} (n={u['n_ctx_has_answer']})   "
                  f"not-in-context {u['acc_given_no_answer_in_ctx']:.3f} (n={u['n_ctx_lacks_answer']})")
        if all(x in R for x in ("bm25", "dense", "hybrid")):
            print("\n-- retriever head-to-head (non-yes/no, containment acc) --")
            for x, y in (("bm25", "dense"), ("bm25", "hybrid"), ("dense", "hybrid")):
                r = A.compare_pair(R[x], R[y])
                print(f"  {y} - {x}: {r['mean_diff']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] p_mcnemar={r['p_mcnemar']:.3g} MDE80={r['mde80']:.3f}")
    df = pd.DataFrame(table).round(4)
    out = os.path.join(a.folder, f"step2_table_{a.dataset}_n{a.n}_p{a.pool}.csv")
    df.to_csv(out, index=False)
    print("\n", df.to_string(index=False), f"\n\nsaved {out}")


if __name__ == "__main__":
    main()

Overwriting analyze_step2.py


In [10]:
%%writefile analyze_step3.py
"""Read a runs3/ folder and print the Step 3 tables.

  python analyze_step3.py runs3 --dataset hotpot

Tables: (1) number of passages k, (2) where the gold passages sit, (3) prompt variants,
(4) error taxonomy and question type, (5) model families. Everything paired by question id, non-yes/no only.
"""
from __future__ import annotations

import argparse
import glob
import os
import re

import numpy as np
import pandas as pd

import slmrag_analysis as A

SET = ["oracle_noise_first", "oracle_noise_last", "oracle_noise", "oracle_rev", "oracle", "closed", "random", "bm25", "dense", "hybrid"]
PAT = re.compile(rf"^(?P<ds>\w+?)__(?P<model>.+?)__(?P<setting>{'|'.join(SET)})_k(?P<k>\d+)_n(?P<n>\d+)_p(?P<p>\d+)_v(?P<v>\w+)_b(?P<b>\d+)\.jsonl$")


def index(folder, dataset):
    idx = {}
    for f in glob.glob(os.path.join(folder, f"{dataset}__*.jsonl")):
        m = PAT.match(os.path.basename(f))
        if m:
            d = m.groupdict()
            idx[(d["model"], d["setting"], int(d["k"]), d["v"], int(d["b"]), int(d["n"]))] = f
    return idx


def acc(rows, key="acc", nonyn=True):
    v = [r[key] for r in rows.values() if not (nonyn and r.get("yn_q"))]
    return float(np.mean(v)) if v else float("nan")


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("folder")
    ap.add_argument("--dataset", default="hotpot")
    a = ap.parse_args(argv)
    idx = index(a.folder, a.dataset)
    if not idx:
        raise SystemExit("no Step 3 runs found in this folder")
    models = sorted({k[0] for k in idx})
    cache = {}
    load = lambda key: cache.setdefault(key, A.load_rows(idx[key]))
    out = []

    def find(model, setting, k=None, v="base", b=None):
        c = [key for key in idx if key[0] == model and key[1] == setting and key[3] == v
             and (k is None or key[2] == k) and (b is None or key[4] == b)]
        return sorted(c, key=lambda t: (-t[5], t[2], t[4]))

    print("=" * 90, "\n(1) HOW MANY PASSAGES? (dense retrieval; budget large so nothing is cut)")
    for m in models:
        ks = sorted({key[2] for key in find(m, "dense")})
        if len(ks) < 2:
            continue
        print(f"-- {m}")
        for k in ks:
            key = find(m, "dense", k)[0]
            R = load(key)
            u = A.utilization(R)
            print(f"   k={k:2d}  acc={acc(R):.3f}  gold-string in prompt {u['share_ctx_has_answer']:.1%}  "
                  f"acc when in prompt {u['acc_given_answer_in_ctx']:.3f}  when not {u['acc_given_no_answer_in_ctx']:.3f}")
            out.append({"table": "k_sweep", "model": m, "k": k, "acc_nonyn": acc(R), **u})

    print("=" * 90, "\n(2) WHERE SIT THE GOLD PASSAGES? (k=5, paired against plain 'oracle' with gold passages only)")
    for m in models:
        base_key = find(m, "oracle", 5)
        if not base_key:
            continue
        base = load(base_key[0])
        print(f"-- {m}   oracle acc={acc(base):.3f}")
        for s in ("oracle_rev", "oracle_noise", "oracle_noise_first", "oracle_noise_last"):
            c = find(m, s, 5)
            if not c:
                continue
            R = load(c[0])
            r = A.compare_pair(base, R)
            print(f"   {s:19s} acc={acc(R):.3f}  diff vs oracle {r['mean_diff']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] p={r['p_mcnemar']:.3g}")
            out.append({"table": "position", "model": m, "setting": s, "acc_nonyn": acc(R), "diff_vs_oracle": r["mean_diff"]})

    print("=" * 90, "\n(3) DOES THE PROMPT CHANGE THINGS? (paired against the same setting with the base prompt)")
    for m in models:
        for s in ("closed", "oracle", "dense", "random"):
            bk = find(m, s, 5, "base")
            if not bk:
                continue
            base = load(bk[0])
            row = f"   {m:28s} {s:7s} base={acc(base):.3f}"
            for v in ("own", "cot"):
                c = find(m, s, 5, v)
                if c:
                    R = load(c[0])
                    r = A.compare_pair(base, R)
                    row += f" | {v}={acc(R):.3f} ({r['mean_diff']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] p={r['p_mcnemar']:.2g})"
                    out.append({"table": "variant", "model": m, "setting": s, "variant": v, "acc_nonyn": acc(R), "diff_vs_base": r["mean_diff"]})
            print(row)
    print("   lost / rescued relative to the BASE closed-book answers:")
    for m in models:
        cb = find(m, "closed", 5, "base")
        if not cb:
            continue
        closed = load(cb[0])
        for s in ("oracle", "dense", "random"):
            for v in ("base", "own", "cot"):
                c = find(m, s, 5, v)
                if c:
                    d = A.distraction_rescue(closed, load(c[0]))
                    print(f"   {m:28s} {s:7s} {v:4s} lost {d.get('lost', float('nan')):.1%}  rescued {d.get('rescued', float('nan')):.1%}")

    print("=" * 90, "\n(4) WHAT DOES A WRONG ANSWER LOOK LIKE? (oracle, base prompt, k=5)")
    for m in models:
        c = find(m, "oracle", 5, "base")
        if not c:
            continue
        R = load(c[0])
        t = A.error_taxonomy(R)
        print(f"-- {m}: " + "  ".join(f"{k}={v:.1%}" for k, v in t.items() if k != "n"))
        for key in ("qtype", "level"):
            g = A.group_acc(R, key)
            if g:
                print(f"   by {key}: " + "  ".join(f"{k}={v[0]:.3f} (n={v[1]})" for k, v in g.items()))
        out.append({"table": "taxonomy", "model": m, **t})

    print("=" * 90, "\n(5) MODEL FAMILIES / SIZES (base prompt, k=5)")
    rows = []
    for m in models:
        row = {"model": m}
        for s in ("closed", "oracle", "dense", "random"):
            c = find(m, s, 5, "base")
            if c:
                row[s] = round(acc(load(c[0])), 3)
        if len(row) > 1:
            rows.append(row)
    if rows:
        print(pd.DataFrame(rows).to_string(index=False))
    df = pd.DataFrame(out)
    path = os.path.join(a.folder, f"step3_tables_{a.dataset}.csv")
    df.to_csv(path, index=False)
    print("\nsaved", path)


if __name__ == "__main__":
    main()

Overwriting analyze_step3.py


In [11]:
%%writefile test_core.py
import numpy as np
import pytest

import slmrag_core as C


# ---- metrics -------------------------------------------------------------
def test_normalize():
    assert C.normalize_answer("The  Eiffel Tower!") == "eiffel tower"


def test_em_f1_aliases():
    golds = ["Barack Obama", "Obama"]
    assert C.exact_match("obama", golds) == 1.0
    assert C.f1_score("President Barack Obama", golds) == pytest.approx(0.8)  # 2*(2/3*1)/(2/3+1)
    assert C.f1_score("Paris", golds) == 0.0


def test_contains_token_boundary():
    assert C.contains_answer("He was born in Paris, France.", ["Paris"])
    assert not C.contains_answer("The party was fun", ["art"])
    assert not C.contains_answer("anything", [""])


def test_label_extraction_and_metrics():
    assert C.extract_label("Yes.") == "yes"
    assert C.extract_label("I am not sure") is None
    m = C.label_metrics(["yes", "no", None, "maybe"], ["yes", "no", "yes", "no"])
    assert m["accuracy"] == 0.5
    assert 0 <= m["macro_f1"] <= 1


# ---- generation cleaning (the root cause of the F1 floor) ----------------
@pytest.mark.parametrize("raw,expected", [
    ("Paris\nQuestion: What is the capital of Spain?\nAnswer: Madrid", "Paris"),
    ("Answer: Barack Obama.", "Barack Obama"),
    ("  \"Paris\"  ", "Paris"),
    ("Paris Question: who wrote Hamlet? Answer: Shakespeare", "Paris"),
    ("Final answer: 1969", "1969"),
    ("Franklin D. Roosevelt", "Franklin D. Roosevelt"),
])
def test_clean_generation(raw, expected):
    assert C.clean_generation(raw) == expected


def test_clean_generation_max_words():
    assert C.clean_generation("one two three four", max_words=2) == "one two"


def test_rambling_output_f1_floor_reproduced():
    """A correct answer followed by fabricated Q/A pairs scores near zero raw, ~1 cleaned."""
    raw = "Paris\n" + "\n".join(f"Question: what is item {i}?\nAnswer: thing number {i} is here" for i in range(6))
    assert C.f1_score(raw, "Paris") < 0.1
    assert C.f1_score(C.clean_generation(raw), "Paris") == 1.0


# ---- prompts -------------------------------------------------------------
def test_truncate_passages_keeps_whole_passages():
    ps = ["a b c", "d e f", "g h i"]
    assert C.truncate_passages(ps, 6) == ["a b c", "d e f"]
    assert C.truncate_passages(["a b c d e"], 3) == ["a b c"]


def test_build_messages_closed_vs_rag():
    closed = C.build_messages("Who?", None)
    rag = C.build_messages("Who?", ["p1", "p2"])
    assert "Context" not in closed[1]["content"]
    assert "[2] p2" in rag[1]["content"]
    assert "yes, no, or maybe" in C.build_messages("q", None, "yesno")[0]["content"]


# ---- retrieval -----------------------------------------------------------
DOCS = [
    "Paris is the capital of France and a major European city",
    "Madrid is the capital of Spain",
    "The mitochondria is the powerhouse of the cell",
    "Berlin is the capital of Germany",
]


def test_bm25_ranks_relevant_first():
    bm = C.BM25(DOCS)
    assert bm.search("capital of France", 2)[0][0] == 0
    assert bm.search("mitochondria cell", 1)[0][0] == 2
    assert bm.search("zzzz unknown", 2) is not None  # no crash on OOV


def test_dense_with_fake_encoder():
    vocab = {w: i for i, w in enumerate(sorted({t for d in DOCS for t in C.tokenize(d)} | {"france"}))}

    def enc(texts):
        out = np.zeros((len(texts), len(vocab)))
        for i, t in enumerate(texts):
            for w in C.tokenize(t):
                if w in vocab:
                    out[i, vocab[w]] += 1
        return out

    d = C.DenseRetriever(DOCS, enc)
    assert d.search("capital france", 1)[0][0] == 0


def test_rrf_and_hybrid():
    fused = C.rrf([[1, 2, 3], [3, 1, 4]], top=3)
    assert fused[0][0] == 1
    class A:  # stub retrievers
        def search(self, q, k): return [(0, 1.0), (1, 0.5)]
    class B:
        def search(self, q, k): return [(1, 1.0), (0, 0.5)]
    assert len(C.HybridRRF(A(), B()).search("x", 2)) == 2


def test_retrieval_metrics():
    ret = [["no", "Paris is here", "x"], ["nothing", "nothing", "nothing"]]
    m = C.retrieval_metrics(ret, [["Paris"], ["Rome"]], ks=(1, 3))
    assert m["hit@1"] == 0.0 and m["hit@3"] == 0.5 and m["mrr@3"] == pytest.approx(0.25)


def test_chunking():
    assert C.chunk_words("a b c d e", 2, 0) == ["a b", "c d", "e"]
    assert C.chunk_words("", 4) == []


def test_id_recall():
    assert C.id_recall_at_k([[1, 2, 3]], [[1, 9]], 3) == 0.5


# ---- statistics ----------------------------------------------------------
def test_bootstrap_ci_covers_mean():
    x = np.random.default_rng(1).normal(0.5, 0.2, 400)
    m, lo, hi = C.bootstrap_ci(x, B=2000)
    assert lo < m < hi and hi - lo < 0.1


def test_paired_compare_detects_shift_and_null():
    rng = np.random.default_rng(0)
    a = rng.normal(0, 1, 300)
    out = C.paired_compare(a + 0.3, a + rng.normal(0, 0.3, 300), B=1000)
    assert out["p_t"] < 0.001 and out["ci_lo"] > 0
    null = C.paired_compare(a, a + rng.normal(0, 0.3, 300), B=1000)
    assert null["p_t"] > 0.01


def test_paired_compare_identical_inputs():
    x = np.ones(50)
    out = C.paired_compare(x, x, binary=True, B=200)
    assert out["p_t"] == 1.0 and out["p_wilcoxon"] == 1.0 and out["p_mcnemar"] == 1.0


def test_mcnemar_exact():
    a = np.array([1] * 20 + [0] * 5 + [1] * 10)
    b = np.array([0] * 20 + [1] * 5 + [1] * 10)
    out = C.paired_compare(a, b, binary=True, B=200)
    assert out["discordant"] == 25 and out["p_mcnemar"] < 0.01


def test_holm_monotone_and_bounded():
    adj = C.holm([0.01, 0.04, 0.03, 0.5])
    assert adj[0] == pytest.approx(0.04) and max(adj) <= 1.0
    assert adj[1] >= adj[2] >= adj[0]


def test_power_matches_manuscript_headline():
    """Manuscript: dz=0.124, diff=0.0059 => SD(diff)~0.0476; at n=300 the MDE exceeds the observed effect."""
    sd = 0.0059 / 0.124
    assert C.mde_paired(300, sd) > 0.0059
    assert C.n_required(0.0059, sd) > 300


def test_kendall():
    tau, p = C.kendall_tau([1, 2, 3, 4], [1, 2, 3, 4])
    assert tau == pytest.approx(1.0)


# ---- evaluate() end to end with a fake rambling model ---------------------
def _records():
    return [
        {"id": "1", "question": "Capital of France?", "answers": ["Paris"], "task": "qa"},
        {"id": "2", "question": "Capital of Spain?", "answers": ["Madrid"], "task": "qa"},
    ]


def test_evaluate_cleaning_and_summary():
    def fake_gen(prompts):
        outs = []
        for p in prompts:
            ans = "Paris" if "Question: Capital of France" in p[1]["content"] else "Madrid"
            outs.append((ans + "\nQuestion: next?\nAnswer: blah blah blah blah", False))
        return outs

    rows, s = C.evaluate(_records(), fake_gen, retrieve_fn=lambda r: DOCS[:2])
    assert s["f1"] == 1.0 and s["acc"] == 1.0 and s["raw_f1"] < 0.5
    assert rows[0]["ctx_has_answer"] is True
    assert any("post-processing" in w for w in C.sanity_warnings({"x": s}))


def test_evaluate_yesno():
    recs = [{"id": "p1", "question": "q?", "label": "yes", "task": "yesno"},
            {"id": "p2", "question": "q?", "label": "no", "task": "yesno"}]
    rows, s = C.evaluate(recs, lambda ps: [("Yes.", False), ("maybe", False)])
    assert s["yn_accuracy"] == 0.5 and s["yn_unparsed"] == 0.0


def test_sanity_flags_runaway_and_oracle_regression():
    good = {"acc": 0.4, "hit_max_rate": 0.0, "empty_rate": 0.0, "mean_pred_words": 3}
    bad = {"acc": 0.3, "hit_max_rate": 0.4, "empty_rate": 0.0, "mean_pred_words": 20}
    w = C.sanity_warnings({"closed": good, "oracle": bad})
    assert any("max_new_tokens" in x for x in w)
    assert any("oracle" in x for x in w)


# ---- data transforms (schema assumed from the HF dataset cards; verify with `slmrag_data.py --check`) ----
import slmrag_data as D


def test_hotpot_record_gold_passages():
    row = {"id": "abc", "question": "Who?", "answer": "Bob",
           "context": {"title": ["A", "B", "C"], "sentences": [["a1.", "a2."], ["b1."], ["c1."]]},
           "supporting_facts": {"title": ["B", "A", "B"], "sent_id": [0, 1, 0]}}
    r = D.hotpot_record(row)
    assert r["gold_ids"] == ["B", "A"] and r["gold_passages"][0].startswith("B: b1.")
    assert r["answers"] == ["Bob"] and r["task"] == "qa"


def test_pubmed_and_nq_records():
    pm = D.pubmed_record({"pubid": 7, "question": "Does X?", "final_decision": "Maybe",
                          "context": {"contexts": ["one.", "two."]}})
    assert pm["label"] == "maybe" and pm["gold_passages"] == ["one. two."] and pm["task"] == "yesno"
    nq = D.nq_record(3, {"question": "who sings x", "answer": ["Reba", "Reba McEntire"]})
    assert nq["question"] == "who sings x?" and len(nq["answers"]) == 2

Overwriting test_core.py


In [12]:
%%writefile test_step2.py
import json
import re
import sys
import types

import numpy as np
import pytest

import slmrag_core as C
import slmrag_analysis as A
import slmrag_data as D
import run_step2 as R2


def fake_encoder(vocab_words):
    vocab = {w: i for i, w in enumerate(sorted(vocab_words))}

    def enc(texts):
        out = np.zeros((len(texts), len(vocab)))
        for i, t in enumerate(texts):
            for w in C.tokenize(t):
                if w in vocab:
                    out[i, vocab[w]] += 1
        return out
    return enc


def make_data(n=30):
    recs = [{"id": f"hp-{i}", "question": f"Capital of C{i}?", "task": "qa", "answers": [f"Town{i}"],
             "gold_passages": [f"C{i}: the capital of C{i} is Town{i}."], "gold_ids": [f"C{i}"]} for i in range(n)]
    pool = [{"id": f"C{i}", "text": f"C{i}: the capital of C{i} is Town{i}."} for i in range(n)]
    pool += [{"id": f"X{i}", "text": f"X{i}: unrelated filler about topic{i} and nothing else"} for i in range(40)]
    return recs, pool


# ---- core additions -------------------------------------------------------
def test_dense_search_batch_equals_single_and_cached_embeddings():
    recs, pool = make_data()
    texts = [p["text"] for p in pool]
    enc = fake_encoder({t for x in texts for t in C.tokenize(x)} | {"capital"})
    d = C.DenseRetriever(texts, enc)
    qs = [r["question"] for r in recs[:5]]
    batch = d.search_batch(qs, 3, block=2)
    assert [[i for i, _ in b] for b in batch] == [[i for i, _ in d.search(q, 3)] for q in qs]
    d2 = C.DenseRetriever(texts, enc, embeddings=d.E)
    assert d2.search(qs[0], 1)[0][0] == d.search(qs[0], 1)[0][0] == 0
    with pytest.raises(AssertionError):
        C.DenseRetriever(texts[:5], enc, embeddings=d.E)


def test_yesno_detector_and_gold_metrics():
    assert C.is_yesno_gold(["yes"]) and C.is_yesno_gold(["No."]) and not C.is_yesno_gold(["Paris"])
    ids = [["a", "b", "c"], ["a", "x", "y"]]
    gold = [["a", "b"], ["a", "b"]]
    assert C.all_gold_at_k(ids, gold, 3) == 0.5 and C.all_gold_at_k(ids, gold, 1) == 0.0
    assert C.id_recall_at_k(ids, gold, 3) == 0.75


def test_calibration_removes_constant_yes_bias_without_labels():
    rng = np.random.default_rng(0)
    labels = (["yes"] * 60 + ["no"] * 30 + ["maybe"] * 10)
    probs = []
    for l in labels:  # model that is informative but adds a big constant bias towards 'yes'
        base = {"yes": 0.2, "no": 0.2, "maybe": 0.2}
        base[l] += 0.25
        base["yes"] += 0.35
        z = sum(base.values())
        probs.append({k: v / z for k, v in base.items()})
    raw = [max(C.LABELS, key=lambda k: p[k]) for p in probs]
    cal = C.calibrate_labels(probs)
    acc = lambda pr: np.mean([a == b for a, b in zip(pr, labels)])
    assert raw.count("yes") == 100            # raw argmax says yes to everything
    assert acc(cal) > acc(raw)                # prior matching recovers signal
    assert cal.count("no") > 0 and cal.count("maybe") > 0


def test_evaluate_label_fn_and_summary_fields():
    recs = [{"id": str(i), "question": "q?", "task": "yesno", "label": l, "answers": [], "gold_passages": ["ctx"]}
            for i, l in enumerate(["yes"] * 6 + ["no"] * 3 + ["maybe"])]
    def label_fn(prompts):
        return [{"yes": .6, "no": .3, "maybe": .1}] * len(prompts)
    def gen_fn(prompts):
        raise AssertionError("generation must not be used when label_fn is given")
    rows, s = C.evaluate(recs, gen_fn, None, label_fn=label_fn)
    assert s["yn_accuracy"] == 0.6 and s["yn_majority_acc"] == 0.6 and s["yn_pred_yes"] == 1.0
    assert "yn_cal_accuracy" in s and "probs" in rows[0] and rows[0]["pred_label"] == "yes"


def test_summarize_nonyn_split():
    recs = [{"id": "1", "question": "q", "answers": ["yes"], "task": "qa"},
            {"id": "2", "question": "q", "answers": ["Paris"], "task": "qa"},
            {"id": "3", "question": "q", "answers": ["Rome"], "task": "qa"}]
    gen = lambda ps: [("yes", False), ("Paris", False), ("wrong", False)]
    rows, s = C.evaluate(recs, gen)
    assert [r["yn_q"] for r in rows] == [True, False, False]
    assert s["acc"] == pytest.approx(2 / 3) and s["acc_nonyn"] == 0.5 and s["n_nonyn"] == 2


# ---- data -----------------------------------------------------------------
def test_build_pool_dev_first_then_extra_dedup_and_cap():
    dev = [("A", "A: a"), ("B", "B: b"), ("A2", "A: a")]
    extra = [("C", "C: c"), ("D", "D: d"), ("E", "E: e")]
    assert [p["id"] for p in D.build_pool(dev, extra, 0)] == ["A", "B"]          # dev only, de-duplicated
    assert [p["id"] for p in D.build_pool(dev, extra, 4)] == ["A", "B", "C", "D"]  # capped at pool_size


# ---- contexts and controls --------------------------------------------------
def test_random_and_oracle_noise_contexts():
    recs, pool = make_data()
    r = recs[3]
    rnd = R2.make_context_cache("random", [r], pool, {}, 5, 42)[r["id"]]
    assert len(rnd) == 5 and r["gold_passages"][0] not in rnd and len(set(rnd)) == 5
    on = R2.make_context_cache("oracle_noise", [r], pool, {}, 5, 42)[r["id"]]
    assert len(on) == 5 and r["gold_passages"][0] in on
    assert on == R2.make_context_cache("oracle_noise", [r], pool, {}, 5, 42)[r["id"]]      # deterministic
    assert on != R2.make_context_cache("oracle_noise", [r], pool, {}, 5, 7)[r["id"]]      # seed matters
    assert R2.make_context_cache("closed", [r], pool, {}, 5, 42) is None


def test_compute_retrieval_cache_and_report(tmp_path):
    recs, pool = make_data()
    enc = fake_encoder({t for p in pool for t in C.tokenize(p["text"])} | {"capital"})
    retr = R2.compute_retrieval(pool, recs, "fake-e5", str(tmp_path), "t", encode_fn=enc)
    assert set(retr) == {"bm25", "dense", "hybrid"} and len(retr["bm25"]["hp-3"]) == R2.TOP
    assert retr["bm25"]["hp-3"][0] == 3 and retr["hybrid"]["hp-3"][0] == 3
    # second call must hit the cache and never touch the encoder
    def boom(_):
        raise AssertionError("encoder used on cache hit")
    again = R2.compute_retrieval(pool, recs, "fake-e5", str(tmp_path), "t", encode_fn=boom)
    assert again == retr
    rep = R2.retrieval_report(retr, pool, recs)
    assert rep["bm25"]["hit@1"] > 0.9 and rep["bm25"]["id_recall@5"] > 0.9 and rep["bm25"]["all_gold@5"] > 0.9


# ---- analysis -----------------------------------------------------------------
def rows_of(accs, ctx=None, yn=None):
    return {str(i): {"id": str(i), "acc": float(a), "f1": float(a), "ctx_has_answer": bool(ctx[i]) if ctx else False,
                     "yn_q": bool(yn[i]) if yn else False} for i, a in enumerate(accs)}


def test_distraction_rescue_and_utilization_and_gain():
    base = rows_of([1] * 50 + [0] * 50)
    cur = rows_of([1] * 40 + [0] * 10 + [1] * 25 + [0] * 25)       # loses 10/50, rescues 25/50
    d = A.distraction_rescue(base, cur)
    assert d["lost"] == pytest.approx(0.2) and d["rescued"] == pytest.approx(0.5)
    assert d["n_base_right"] == 50 and d["n_base_wrong"] == 50
    base_yn = rows_of([1, 1, 0, 0], yn=[1, 0, 0, 0])
    assert A.distraction_rescue(base_yn, rows_of([1] * 4, yn=[1, 0, 0, 0]))["n"] == 3   # yes/no question dropped

    u = A.utilization(rows_of([1, 1, 0, 0, 0, 0], ctx=[1, 1, 1, 1, 0, 0]))
    assert u["acc_given_answer_in_ctx"] == 0.5 and u["acc_given_no_answer_in_ctx"] == 0.0

    g = A.gain_table(base, {"oracle": rows_of([1] * 90 + [0] * 10), "random": rows_of([1] * 48 + [0] * 52)})
    assert g[0]["mean_diff"] > 0.3 and g[0]["p_holm"] < 1e-6 and g[1]["p_holm"] >= g[1]["p"] - 1e-12


# ---- end to end with a fake model (no torch / network) ---------------------------
def test_run_step2_end_to_end_and_resume(tmp_path, monkeypatch, capsys):
    recs, pool = make_data()
    monkeypatch.setattr(R2.D, "load_hotpot_open", lambda n, seed, pool_size: (recs[:n or None], pool))
    enc = fake_encoder({t for p in pool for t in C.tokenize(p["text"])} | {"capital"})
    monkeypatch.setattr(R2.C, "make_st_encoder", lambda name: enc)

    calls = {"n": 0}

    def fake_gen(prompts):
        calls["n"] += 1
        outs = []
        for p in prompts:
            c = p[1]["content"]
            q = re.search(r"Question: Capital of (C\d+)\?", c).group(1)
            outs.append((f"Town{q[1:]}\nQuestion: more?" if f"capital of {q} is" in c else "unknown", False))
        return outs

    fake = types.ModuleType("slmrag_hf")
    fake.load_model = lambda name, load_4bit=False: (None, None)
    fake.make_generate_fn = lambda tok, model, mnt, bs: fake_gen
    fake.make_label_fn = lambda tok, model: None
    monkeypatch.setitem(sys.modules, "slmrag_hf", fake)

    argv = ["--dataset", "hotpot", "--n", "30", "--k", "3", "--pool_size", "70", "--cache_dir", str(tmp_path / "c"),
            "--settings", "closed,oracle,oracle_noise,random,bm25,dense,hybrid", "--out", str(tmp_path / "o"), "--resume"]
    R2.main(argv)
    allj = json.load(open(next((tmp_path / "o").glob("*__ALL_*.json"))))
    assert allj["closed"]["acc"] == 0.0 and allj["oracle"]["acc"] == 1.0
    assert allj["oracle_noise"]["acc"] == 1.0 and allj["random"]["acc"] == 0.0
    assert allj["bm25"]["acc"] == 1.0 and allj["dense"]["acc"] == 1.0 and allj["hybrid"]["ret_all_gold@5"] == 1.0
    first = calls["n"]
    capsys.readouterr()
    R2.main(argv)                                   # resume: nothing is regenerated
    assert calls["n"] == first and "resumed from disk" in capsys.readouterr().out

    # analysis script over the files just written
    import analyze_step2
    monkeypatch.setattr(sys, "argv", ["a", str(tmp_path / "o"), "--n", "30", "--pool", str(len(pool)), "--k", "3"])
    analyze_step2.main()
    out = capsys.readouterr().out
    assert "distraction" in out and "utilisation" in out and "retriever head-to-head" in out
    assert list((tmp_path / "o").glob("step2_table_*.csv"))
    monkeypatch.setattr(sys, "argv", ["a", str(tmp_path / "o"), "--k", "3"])      # n / pool auto-detected
    analyze_step2.main()
    assert "auto-detected n=30" in capsys.readouterr().out


def test_retrieval_only_never_loads_a_model(tmp_path, monkeypatch):
    recs, pool = make_data()
    monkeypatch.setattr(R2.D, "load_hotpot_open", lambda n, seed, pool_size: (recs, pool))
    monkeypatch.setattr(R2.C, "make_st_encoder", lambda name: fake_encoder({t for p in pool for t in C.tokenize(p["text"])}))
    monkeypatch.delitem(sys.modules, "slmrag_hf", raising=False)
    R2.main(["--dataset", "hotpot", "--retrieval_only", "--cache_dir", str(tmp_path / "c"), "--out", str(tmp_path / "o")])
    assert list((tmp_path / "o").glob("retrieval__*.json")) and "slmrag_hf" not in sys.modules

Overwriting test_step2.py


In [13]:
%%writefile test_step3.py
import json, re, sys, types
import pytest
import slmrag_core as C
import slmrag_analysis as A
import run_step2 as R2
import run_step3 as R3
from test_step2 import make_data, fake_encoder


def test_extract_final_and_variants():
    assert C.extract_final("Paris is the capital.\nFinal answer: Paris.") == ("Paris", True)
    assert C.extract_final("x\nFinal answer: A\nFinal answer: B") == ("B", True)       # last marker wins
    assert C.extract_final("thinking...\nParis") == ("Paris", False)                   # fallback: last line
    assert C.extract_final("") == ("", False)
    base = C.build_messages("Q?", ["P"], "qa", "base")[0]["content"]
    own = C.build_messages("Q?", ["P"], "qa", "own")[0]["content"]
    cot = C.build_messages("Q?", ["P"], "qa", "cot")[0]["content"]
    assert base != own != cot and "own knowledge" in own and "Final answer" in cot
    assert C.build_messages("Q?", None, "yesno", "cot")[0]["content"] == C.SYSTEM_YESNO   # PubMed ignores variants


def test_evaluate_cot_and_pred_in_ctx():
    recs = [{"id": "1", "question": "Capital of France?", "answers": ["Paris"], "task": "qa", "qtype": "bridge", "level": "hard"},
            {"id": "2", "question": "Capital of Italy?", "answers": ["Rome"], "task": "qa"}]
    ctx = {"1": ["Berlin is in Germany. Lyon is in France."], "2": ["Rome is the capital of Italy."]}
    gen = lambda ps: [("Reasoning...\nFinal answer: Lyon", False), ("It is Rome.\nFinal answer: Rome.", False)]
    rows, s = C.evaluate(recs, gen, lambda r: ctx[r["id"]], variant="cot")
    assert [r["pred"] for r in rows] == ["Lyon", "Rome"]
    assert rows[0]["pred_in_ctx"] and rows[0]["acc"] == 0.0 and rows[1]["acc"] == 1.0
    assert rows[0]["qtype"] == "bridge" and s["final_found_rate"] == 1.0
    tax = A.error_taxonomy({r["id"]: r for r in rows})
    assert tax["correct"] == 0.5 and tax["wrong_copied"] == 0.5
    assert A.group_acc({r["id"]: r for r in rows}, "level") == {"hard": (0.0, 1)}


def test_error_taxonomy_abstain_and_other():
    mk = lambda acc, pred, inctx: {"acc": acc, "pred": pred, "pred_in_ctx": inctx, "task": "qa"}
    t = A.error_taxonomy({"a": mk(0, "Not applicable", False), "b": mk(0, "Zed", False), "c": mk(0, "", False), "d": mk(1, "ok", True)})
    assert t["abstain"] == t["wrong_other"] == t["empty"] == t["correct"] == 0.25


def test_position_contexts():
    recs, pool = make_data()
    r = recs[2]
    g = r["gold_passages"][0]
    first = R3.make_context_cache("oracle_noise_first", [r], pool, {}, 5, 1)[r["id"]]
    last = R3.make_context_cache("oracle_noise_last", [r], pool, {}, 5, 1)[r["id"]]
    assert first[0] == g and last[-1] == g and len(first) == len(last) == 5 and g not in first[1:]
    two = {"id": "x", "gold_passages": ["A", "B"]}
    assert R3.make_context_cache("oracle_rev", [two], pool, {}, 5, 1)["x"] == ["B", "A"]
    assert R3.make_context_cache("oracle", [two], pool, {}, 5, 1)["x"] == ["A", "B"]      # falls through to Step 2


def test_top_param_changes_cache_and_length(tmp_path):
    recs, pool = make_data()
    enc = fake_encoder({t for p in pool for t in C.tokenize(p["text"])} | {"capital"})
    a = R2.compute_retrieval(pool, recs, "fake", str(tmp_path), "t", encode_fn=enc, top=10)
    b = R2.compute_retrieval(pool, recs, "fake", str(tmp_path), "t", encode_fn=enc, top=20)
    assert len(a["dense"]["hp-1"]) == 10 and len(b["dense"]["hp-1"]) == 20 and len(b["hybrid"]["hp-1"]) == 20


def test_run_step3_end_to_end_and_analysis(tmp_path, monkeypatch, capsys):
    recs, pool = make_data()
    for i, r in enumerate(recs):
        r["qtype"], r["level"] = ("bridge" if i % 2 else "comparison"), "hard"
    monkeypatch.setattr(R3.D, "load_hotpot_open", lambda n, seed, pool_size: (recs[:n or None], pool))
    enc = fake_encoder({t for p in pool for t in C.tokenize(p["text"])} | {"capital"})
    monkeypatch.setattr(R3.R2.C, "make_st_encoder", lambda name: enc)

    def fake_gen(prompts):
        outs = []
        for p in prompts:
            sysm, c = p[0]["content"], p[1]["content"]
            q = re.search(r"Question: Capital of (C\d+)\?", c).group(1)
            ans = f"Town{q[1:]}" if f"capital of {q} is" in c else "unknown"
            outs.append((f"Let me think.\nFinal answer: {ans}" if "Final answer" in sysm else ans, False))
        return outs

    fake = types.ModuleType("slmrag_hf")
    fake.load_model = lambda name, load_4bit=False: (None, None)
    fake.make_generate_fn = lambda tok, model, mnt, bs: fake_gen
    monkeypatch.setitem(sys.modules, "slmrag_hf", fake)

    common = ["--dataset", "hotpot", "--n", "30", "--pool_size", "70", "--cache_dir", str(tmp_path / "c"), "--out", str(tmp_path / "o"), "--resume"]
    R3.main(common + ["--settings", "closed,oracle,oracle_rev,oracle_noise,oracle_noise_first,oracle_noise_last,random,dense", "--k", "5"])
    R3.main(common + ["--settings", "dense", "--k", "3"])
    R3.main(common + ["--settings", "closed,oracle,dense,random", "--k", "5", "--variant", "cot"])
    R3.main(common + ["--settings", "closed,oracle", "--k", "5", "--variant", "own"])
    capsys.readouterr()
    R3.main(common + ["--settings", "closed,oracle", "--k", "5", "--variant", "own"])
    assert "resumed from disk" in capsys.readouterr().out

    import analyze_step3
    analyze_step3.main([str(tmp_path / "o")])
    out = capsys.readouterr().out
    for head in ("(1) HOW MANY PASSAGES", "(2) WHERE SIT", "(3) DOES THE PROMPT", "(4) WHAT DOES A WRONG", "(5) MODEL FAMILIES"):
        assert head in out
    assert "k= 3" in out and "k= 5" in out and "cot=1.000" in out and "bridge=" in out
    assert list((tmp_path / "o").glob("step3_tables_*.csv"))


def test_bad_setting_rejected(tmp_path):
    with pytest.raises(SystemExit):
        R3.main(["--settings", "oracel", "--out", str(tmp_path)])

Overwriting test_step3.py


## 2. Tests (50 expected to pass) and data check

In [14]:
!python -m pytest -q -p no:cacheprovider test_core.py test_step2.py test_step3.py

..................................................                       [100%]
50 passed in 1.70s


In [15]:
!python slmrag_data.py --check

nq {'id': 'nq-3283', 'question': 'who sold out jesus for 30 pieces of silver?', 'task': 'qa', 'answers': ['Judas Iscariot'], 'gold_passages': [], 'gold_ids': []}
hotpot {'id': 'hp-5a7a567255429941d65f25bd', 'question': 'What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?', 'task': 'qa', 'answers': ['flotilla'], 'gold_passages': ['Iqbal F. Qadir: Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a reti', 'Dwarka: Dwarka () is a small city and a municipality of Devbhoomi Dwarka distric'], 'gold_ids': ['Iqbal F. Qadir', 'Dwarka'], 'qtype': 'bridge', 'level': 'hard'}
pubmed {'id': 'pm-10922093', 'question': 'Does open access endoscopy close the door to an adequately informed patient?', 'task': 'yesno', 'label': 'yes', 'answers': [], 'gold_passages': ['The use of open access endoscopy is increasing. Its effect on the adequacy of pa'], 'gold_ids': ['10922093']}
hotpot pool=66635 paragraphs, pubmed pool=

## 3. Configuration and smoke test

In [16]:
QWEN     = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
FAMILIES = ["HuggingFaceTB/SmolLM2-1.7B-Instruct", "microsoft/Phi-3.5-mini-instruct"]
N, POOL, BUDGET = 2000, 200_000, 4000     # same 2,000 questions as Step 2; BUDGET (words) is large so no passage is cut off
OUT, CACHE = "/kaggle/working/runs3", "/kaggle/working/cache"
COMMON = f"--dataset hotpot --n {N} --pool_size {POOL} --context_budget {BUDGET} --resume --out {OUT} --cache_dir {CACHE}"

In [17]:
!python run_step3.py --dataset hotpot --model {QWEN[0]} --smoke --out {OUT}/smoke --cache_dir {CACHE}

hotpot: 20 questions, pool 66,635 passages, variant=base, k=5
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 264.76it/s]
closed              n=20 acc_nonyn=0.000 em=0.000 words=1.9 final_found=100% hit_max=0.0% (0.0 min)
oracle              n=20 acc_nonyn=0.684 em=0.600 words=2.4 final_found=100% hit_max=0.0% (0.0 min)
oracle_rev          n=20 acc_nonyn=0.632 em=0.600 words=2.2 final_found=100% hit_max=0.0% (0.0 min)
oracle_noise_last   n=20 acc_nonyn=0.526 em=0.500 words=1.9 final_found=100% hit_max=0.0% (0.1 min)


Smoke test: no WARNING lines, and `oracle_noise_last` should not be far below `oracle` on 20 questions. If the `closed` row for this prompt looks odd, stop and send me the output.

## 4. Tests A, B, C on the two Qwen models
The first dense run builds the retrieval cache (same time as Step 2: it embeds 200,000 passages). Later runs reuse it.

In [18]:
# A. number of passages (dense). k=5 also serves as the base for the other tests.
for m in QWEN:
    for k in (5, 1, 3, 10, 20):
        print("=== A", m, "k =", k)
        !python run_step3.py {COMMON} --model {m} --k {k} --settings dense

=== A Qwen/Qwen2.5-1.5B-Instruct k = 5
hotpot: 2000 questions, pool 200,000 passages, variant=base, k=5
retrieval cache hit: /kaggle/working/cache/retr_hotpot_p200000_31d2f8bd6f_da4115ae_intfloat-e5-base-v2_top20.json
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 268.73it/s]
dense: resumed from disk
=== A Qwen/Qwen2.5-1.5B-Instruct k = 1
hotpot: 2000 questions, pool 200,000 passages, variant=base, k=1
retrieval cache hit: /kaggle/working/cache/retr_hotpot_p200000_31d2f8bd6f_da4115ae_intfloat-e5-base-v2_top20.json
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 258.27it/s]
dense: resumed from disk
=== A Qwen/Qwen2.5-1.5B-Instruct k = 3
hotpot: 2000 questions, pool 200,000 passages, variant=base, k=3
retrieval cache hit: /kaggle/working/cache/retr_hotpot_p200000_31d2f8bd6f_da4115ae_intfloat-e5-base-v2_top20.json
[tra

In [19]:
# B. position and order of the gold passages (k = 5)
for m in QWEN:
    print("=== B", m)
    !python run_step3.py {COMMON} --model {m} --k 5 --settings closed,oracle,oracle_rev,oracle_noise,oracle_noise_first,oracle_noise_last,random

=== B Qwen/Qwen2.5-1.5B-Instruct
hotpot: 2000 questions, pool 200,000 passages, variant=base, k=5
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 257.91it/s]
closed: resumed from disk
oracle: resumed from disk
oracle_rev: resumed from disk
oracle_noise: resumed from disk
oracle_noise_first: resumed from disk
oracle_noise_last: resumed from disk
random: resumed from disk
=== B Qwen/Qwen2.5-3B-Instruct
hotpot: 2000 questions, pool 200,000 passages, variant=base, k=5
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 434/434 [00:03<00:00, 127.15it/s]
closed: resumed from disk
oracle: resumed from disk
oracle_rev: resumed from disk
oracle_noise: resumed from disk
oracle_noise_first: resumed from disk
oracle_noise_last: resumed from disk
random              n=2000 acc_nonyn=0.073 em=0.093 words=4.5 final_found=100% hit_max=0.1% (12.7 min)


In [20]:
# C. prompt variants (k = 5). 'cot' writes up to 160 tokens, so it is slower.
for v in ("own", "cot"):
    for m in QWEN:
        print("=== C", v, m)
        !python run_step3.py {COMMON} --model {m} --k 5 --variant {v} --settings closed,oracle,dense,random

=== C own Qwen/Qwen2.5-1.5B-Instruct
hotpot: 2000 questions, pool 200,000 passages, variant=own, k=5
retrieval cache hit: /kaggle/working/cache/retr_hotpot_p200000_31d2f8bd6f_da4115ae_intfloat-e5-base-v2_top20.json
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 252.41it/s]
closed              n=2000 acc_nonyn=0.100 em=0.120 words=2.2 final_found=100% hit_max=0.0% (1.3 min)
oracle              n=2000 acc_nonyn=0.575 em=0.523 words=2.2 final_found=100% hit_max=0.1% (2.6 min)
dense               n=2000 acc_nonyn=0.362 em=0.337 words=2.3 final_found=100% hit_max=0.2% (5.4 min)
random              n=2000 acc_nonyn=0.100 em=0.115 words=2.3 final_found=100% hit_max=0.2% (6.0 min)
=== C own Qwen/Qwen2.5-3B-Instruct
hotpot: 2000 questions, pool 200,000 passages, variant=own, k=5
retrieval cache hit: /kaggle/working/cache/retr_hotpot_p200000_31d2f8bd6f_da4115ae_intfloat-e5-base-v2_top20.json
[transformers] `to

## 5. Other model families (base prompt, k = 5)
If a model fails to load (for example a changed repo name), the loop just moves on to the next one; tell me which and I will swap it.
Gated models (Llama-3.2, Gemma) need a Hugging Face token and are not included here.

In [21]:
for m in FAMILIES:
    print("=== E", m)
    !python run_step3.py {COMMON} --model {m} --k 5 --settings closed,oracle,dense,random

=== E HuggingFaceTB/SmolLM2-1.7B-Instruct
hotpot: 2000 questions, pool 200,000 passages, variant=base, k=5
retrieval cache hit: /kaggle/working/cache/retr_hotpot_p200000_31d2f8bd6f_da4115ae_intfloat-e5-base-v2_top20.json
config.json: 100%|█████████████████████████████| 908/908 [00:00<00:00, 1.52MB/s]
tokenizer_config.json: 100%|███████████████| 3.76k/3.76k [00:00<00:00, 16.2MB/s]
vocab.json: 100%|████████████████████████████| 801k/801k [00:00<00:00, 25.0MB/s]
merges.txt: 100%|█████████████████████████████| 466k/466k [00:00<00:00, 100MB/s]
special_tokens_map.json: 100%|█████████████████| 655/655 [00:00<00:00, 3.88MB/s]
tokenizer.json: 100%|███████████████████████| 2.10M/2.10M [00:00<00:00, 133MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:                       | 19.0MB, 47.2kB/s  
model.safetensors: downloading bytes: ▉                     |  146MB, 9.21MB/s  
model.safetensors: downloading bytes: ███                   |  47

## 6. Tables

In [22]:
!python analyze_step3.py {OUT} --dataset hotpot

(1) HOW MANY PASSAGES? (dense retrieval; budget large so nothing is cut)
-- Qwen-Qwen2-5-1-5B-Instruct
   k= 1  acc=0.280  gold-string in prompt 38.4%  acc when in prompt 0.606  when not 0.078
   k= 3  acc=0.354  gold-string in prompt 65.1%  acc when in prompt 0.528  when not 0.031
   k= 5  acc=0.356  gold-string in prompt 71.6%  acc when in prompt 0.490  when not 0.017
   k=10  acc=0.365  gold-string in prompt 78.5%  acc when in prompt 0.459  when not 0.022
   k=20  acc=0.365  gold-string in prompt 83.6%  acc when in prompt 0.434  when not 0.016
-- Qwen-Qwen2-5-3B-Instruct
   k= 1  acc=0.272  gold-string in prompt 38.4%  acc when in prompt 0.595  when not 0.071
   k= 3  acc=0.363  gold-string in prompt 65.1%  acc when in prompt 0.543  when not 0.026
   k= 5  acc=0.370  gold-string in prompt 71.6%  acc when in prompt 0.511  when not 0.017
   k=10  acc=0.383  gold-string in prompt 78.5%  acc when in prompt 0.486  when not 0.007
   k=20  acc=0.370  gold-string in prompt 83.6%  acc when i

## 7. Package outputs

In [23]:
import shutil
shutil.make_archive("/kaggle/working/step3_outputs", "zip", OUT)
print("saved /kaggle/working/step3_outputs.zip (Output tab after 'Save Version'). Send me that zip.")

saved /kaggle/working/step3_outputs.zip (Output tab after 'Save Version'). Send me that zip.


## Gates before Step 4
- [ ] Smoke test has no WARNING lines
- [ ] For `cot`, `final_found` is above 90% (otherwise the extraction, not the model, is the problem)
- [ ] Every claim in the paper from these tables names its paired CI